# V7 restaurant-turnover research — resumable CPU workflow

This notebook reruns the 12 bounded settings, locks one configuration per family,
confirms them and creates up to two experimental CSVs. It uses the four official
inputs and embeds all its own source and the preserved scored v5 reference.
It does not require previous improvement folders or upload anything to Olympus.

CPU is sufficient for this small dataset. Your screenshot shows approximately
12.7 GB of RAM; a GPU subscription is not required for this implementation. Free
Colab runtime availability varies. For checkpoints to survive a deleted runtime,
enable Google Drive below and grant access yourself. Upload the four official input
CSVs again after a runtime reset; completed, hash-verified folds will be reused.

In [ ]:
import os, sys, json, hashlib, base64, zlib, types, subprocess, importlib.metadata
from pathlib import Path

PACKAGES = {'numpy': '2.5.4', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'catboost': '1.2.10', 'scipy': '1.18.1'}
install = []
for name, version in PACKAGES.items():
    try:
        installed = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:
        install.append(f'{name}=={version}')
if install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *install])
import numpy as np
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'matplotlib'])
    import matplotlib.pyplot as plt
print('Python:', sys.version.split()[0])
print({p: importlib.metadata.version(p) for p in PACKAGES})

In [ ]:
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['MLA_OUTPUT_DIR'] = '/content/drive/MyDrive/MLA_v7/research'
DATA_DIR = Path(os.environ.get('MLA_DATA_DIR', '.')).resolve()
OUTPUT_DIR = Path(os.environ.get('MLA_OUTPUT_DIR', '.')).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REQUIRED = ['Train_dataset.csv', 'Test_dataset.csv', 'Data_Dictionary.csv', 'Sample_Solution.csv']
if not all((DATA_DIR / n).is_file() for n in REQUIRED):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place the four official CSVs here, or set MLA_DATA_DIR.')
    files.upload()
    DATA_DIR = Path.cwd()
INPUT_HASHES = {n: hashlib.sha256((DATA_DIR / n).read_bytes()).hexdigest() for n in REQUIRED}

EXPECTED_INPUT_HASHES = {'Train_dataset.csv': 'ec40ebda74ba0bfbfe1af4417a4266a70eadf6e595500e99e5435e7bdc7ee51f', 'Test_dataset.csv': 'ee4137566adb1c0ce356a8c370e7b8a22f681bde492068efe11346d8d44b4f4d', 'Data_Dictionary.csv': '2d86c00e27448593564a161197b6639ee1eb90c01fa523b67e7a09d2ef405c5e', 'Sample_Solution.csv': '9519ba4d51eb2d59b8a71b054a62e1f4d48b72cf92930c31f5e5393a2cd135fb'}
assert INPUT_HASHES == EXPECTED_INPUT_HASHES, 'Input files differ from the measured experiment.'
print('PASS: four official input files and their SHA256 hashes.')

In [ ]:
SOURCE_FILES = json.loads('{"reference.py":"\\"\\"\\"Frozen v5 predictor, copied verbatim from the preserved implementations.\\"\\"\\"\\nimport numpy as np\\nimport pandas as pd\\nfrom catboost import CatBoostRegressor\\nfrom sklearn.model_selection import KFold, GroupKFold\\nfrom sklearn.compose import ColumnTransformer\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler, SplineTransformer\\nfrom sklearn.linear_model import Ridge\\nfrom sklearn.ensemble import HistGradientBoostingRegressor, ExtraTreesRegressor\\nfrom sklearn.svm import SVR\\n\\n\\nTARGET = \'Annual Turnover\'\\n\\nID = \'Registration Number\'\\n\\nEVENTS = [\'Live Music Rating\',\'Comedy Gigs Rating\',\'Value Deals Rating\',\'Live Sports Rating\']\\n\\nSOCIAL = [\'Facebook Popularity Quotient\',\'Instagram Popularity Quotient\']\\n\\nCORE = SOCIAL + [\'Hygiene Rating\',\'Restaurant Zomato Rating\',\'Live Music Rating\',\'Value for Money\',\'Overall Restaurant Rating\',\'Restaurant City Tier\',\'Cuisine\',\'Restaurant Theme\',\'Restaurant Location\',\'Endoresed By\',\'City\',\'Opening Year\']\\n\\ndef features(raw, variant=\'base\'):\\n    data=raw.drop(columns=[TARGET,ID],errors=\'ignore\').copy()\\n    dates=pd.to_datetime(data.pop(\'Opening Day of Restaurant\'),format=\'%d-%m-%Y\',errors=\'coerce\')\\n    data[\'Opening Year\']=dates.dt.year.astype(float)\\n    data[\'Opening Month\']=dates.dt.month.astype(float)\\n    for c in EVENTS:\\n        data[\'Offers \'+c.replace(\' Rating\',\'\')]=data[c].notna().astype(int)\\n    parts=data[\'Cuisine\'].astype(\'string\').fillna(\'Missing\').str.split(\',\',n=1,expand=True)\\n    data[\'Cuisine 1\']=parts[0].str.strip().str.lower()\\n    data[\'Cuisine 2\']=parts.reindex(columns=[0,1])[1].astype(\'string\').str.strip().str.lower()\\n    for c in [\'Cuisine\',\'City\',\'Restaurant Location\',\'Endoresed By\',\'Restaurant Type\',\'Restaurant Theme\']:\\n        data[c]=data[c].astype(\'string\').str.strip().str.lower().str.replace(r\'\\\\s+\',\' \',regex=True).fillna(\'Missing\')\\n    data[\'City Is Unknown\']=(data[\'City\']==\'-1\').astype(int)\\n    for c in data.select_dtypes(include=[\'object\',\'string\']).columns:\\n        data[c]=data[c].fillna(\'Missing\').astype(str)\\n    if variant in [\'engineered\',\'engineered_drop\']:\\n        data[\'City\']=data[\'City\'].replace({\'bengaluru\':\'bangalore\',\'gurugram\':\'gurgaon\',\'bombay\':\'mumbai\',\'calcutta\':\'kolkata\',\'delhi\':\'new delhi\'})\\n        data[\'Cuisine unordered\']=[\'|\'.join(sorted(v.split(\',\'))) for v in data[\'Cuisine\']]\\n        data[\'Opening Day\']=dates.dt.day.astype(float)\\n        data[\'Opening Days Since 1970\']=(dates-pd.Timestamp(\'1970-01-01\')).dt.days.astype(float)\\n        data[\'Social mean\']=data[SOCIAL].mean(axis=1)\\n        data[\'Social difference\']=data[SOCIAL[0]]-data[SOCIAL[1]]\\n        data[\'Social missing count\']=data[SOCIAL].isna().sum(axis=1)\\n        data[\'Rating missing count\']=data.isna().sum(axis=1)\\n        data[\'Event count\']=data[EVENTS].notna().sum(axis=1)\\n        data[\'Event mean\']=data[EVENTS].mean(axis=1)\\n        data[\'Quality mean\']=data[[\'Food Rating\',\'Hygiene Rating\',\'Overall Restaurant Rating\']].mean(axis=1)\\n        data[\'Social x Zomato\']=data[\'Social mean\']*data[\'Restaurant Zomato Rating\']\\n        data[\'Social x Hygiene\']=data[\'Social mean\']*data[\'Hygiene Rating\']\\n        data[\'Quality x Value\']=data[\'Quality mean\']*data[\'Value for Money\']\\n    if variant in [\'drop_shift\',\'engineered_drop\']:\\n        data=data.drop(columns=[\'Service\',\'Comfortablility\'])\\n    if variant==\'core\': data=data[CORE]\\n    if variant==\'ratings\':\\n        keep=SOCIAL+[\'Resturant Tier\',\'Restaurant Zomato Rating\',\'Restaurant City Tier\',\'Staff Responsivness\',\'Value for Money\',\'Hygiene Rating\',\'Overall Restaurant Rating\']+EVENTS+[\'Lively\',\'Ambience\',\'Privacy\']\\n        data=data[keep]\\n    if variant==\'no_city_date\':\\n        data=data.drop(columns=[\'City\',\'City Is Unknown\',\'Opening Year\',\'Opening Month\'])\\n    return data\\n\\ndef rmse(y,p): return float(np.sqrt(np.mean(np.square(np.asarray(y)-np.asarray(p)))))\\n\\ndef sklearn_model(X,spec,seed):\\n    cats=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist()\\n    nums=[c for c in X if c not in cats]\\n    num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),StandardScaler())\\n    if spec[\'kind\']==\'spline\':\\n        num=make_pipeline(SimpleImputer(strategy=\'median\',add_indicator=True),SplineTransformer(n_knots=4,degree=2,include_bias=False),StandardScaler())\\n    pre=ColumnTransformer([(\'num\',num,nums),(\'cat\',OneHotEncoder(handle_unknown=\'ignore\',min_frequency=10,sparse_output=False),cats)])\\n    kind=spec[\'kind\']\\n    if kind in [\'ridge\',\'spline\']: model=Ridge(alpha=spec.get(\'alpha\',100))\\n    elif kind==\'svr\': model=SVR(C=spec.get(\'C\',20),epsilon=.5,kernel=\'rbf\')\\n    elif kind==\'hist\': model=HistGradientBoostingRegressor(max_iter=300,max_leaf_nodes=10,max_depth=4,learning_rate=.035,l2_regularization=30,min_samples_leaf=60,random_state=seed)\\n    elif kind==\'extra\': model=ExtraTreesRegressor(n_estimators=400,min_samples_leaf=5,max_features=.7,n_jobs=4,random_state=seed)\\n    return make_pipeline(pre,model)\\n\\ndef fit_predict(X,y,train_idx,valid_idx,spec,seed,fixed_iterations=None,return_fitted=False):\\n    ys=np.asarray(y)[train_idx].copy()\\n    if \'cap\' in spec: ys=np.minimum(ys,spec[\'cap\'])\\n    if spec[\'kind\']==\'cat\':\\n        params=dict(loss_function=spec.get(\'loss\',\'RMSE\'),eval_metric=\'RMSE\',iterations=fixed_iterations or spec.get(\'iterations\',1600),learning_rate=.04,depth=spec.get(\'depth\',6),l2_leaf_reg=spec.get(\'l2\',5),random_seed=seed,thread_count=4,verbose=False,allow_writing_files=False)\\n        if \'boosting_type\' in spec: params[\'boosting_type\']=spec[\'boosting_type\']\\n        if spec.get(\'monotone\'): params[\'monotone_constraints\']={**{c:1 for c in SOCIAL+[\'Hygiene Rating\',\'Restaurant Zomato Rating\']},\'Resturant Tier\':-1}\\n        model=CatBoostRegressor(**params)\\n        cat=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist()\\n        stopping={} if fixed_iterations or \'iterations\' in spec else dict(eval_set=(X.iloc[valid_idx],np.asarray(y)[valid_idx]),early_stopping_rounds=120)\\n        model.fit(X.iloc[train_idx],ys,cat_features=cat,**stopping)\\n        p=model.predict(X.iloc[valid_idx]); trees=model.tree_count_\\n    elif spec[\'kind\'] in [\'lgb\',\'xgb\']:\\n        pre=sklearn_model(X,dict(kind=\'ridge\'),seed)[:-1]\\n        ztr=pre.fit_transform(X.iloc[train_idx]); zva=pre.transform(X.iloc[valid_idx])\\n        early=not (fixed_iterations or \'iterations\' in spec)\\n        iterations=fixed_iterations or spec.get(\'iterations\',1400)\\n        if spec[\'kind\']==\'lgb\':\\n            from lightgbm import LGBMRegressor,early_stopping\\n            model=LGBMRegressor(n_estimators=iterations,learning_rate=.025,num_leaves=spec.get(\'leaves\',15),min_child_samples=spec.get(\'leaf_samples\',60),reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,subsample_freq=1,random_state=seed,n_jobs=4,verbosity=-1,objective=spec.get(\'objective\',\'regression\'))\\n            params=dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],eval_metric=\'rmse\',callbacks=[early_stopping(100,verbose=False,first_metric_only=True)]) if early else {}\\n            model.fit(ztr,ys/1e6,**params)\\n            trees=model.best_iteration_ if early else iterations\\n        else:\\n            from xgboost import XGBRegressor\\n            params=dict(n_estimators=iterations,max_depth=spec.get(\'depth\',3),learning_rate=.025,min_child_weight=30,reg_lambda=20,reg_alpha=.1,colsample_bytree=.8,subsample=.8,random_state=seed,n_jobs=4,tree_method=\'hist\',objective=spec.get(\'objective\',\'reg:squarederror\'),eval_metric=\'rmse\',base_score=float(ys.mean()/1e6))\\n            if early: params[\'early_stopping_rounds\']=100\\n            if spec.get(\'objective\')==\'reg:pseudohubererror\': params[\'huber_slope\']=12\\n            model=XGBRegressor(**params)\\n            model.fit(ztr,ys/1e6,**(dict(eval_set=[(zva,np.asarray(y)[valid_idx]/1e6)],verbose=False) if early else {}))\\n            trees=model.best_iteration+1 if early else iterations\\n        p=model.predict(zva)*1e6\\n    else:\\n        model=sklearn_model(X,spec,seed)\\n        model.fit(X.iloc[train_idx],ys/1e6)\\n        p=model.predict(X.iloc[valid_idx])*1e6; trees=0\\n    if return_fitted:\\n        fitted=dict(model=model,preprocessor=pre if spec[\'kind\'] in [\'lgb\',\'xgb\'] else None,scale=1 if spec[\'kind\']==\'cat\' else 1e6)\\n        return np.maximum(p,0),trees,fitted\\n    return np.maximum(p,0),trees\\n\\ndef predict_fitted(fitted,X):\\n    transformed=fitted[\'preprocessor\'].transform(X) if fitted[\'preprocessor\'] is not None else X\\n    return np.maximum(fitted[\'model\'].predict(transformed)*fitted[\'scale\'],0)\\n\\ndef cat_fit(X, y, spec, seed):\\n    params=dict(iterations=spec[\'iterations\'], learning_rate=.04, depth=spec[\'depth\'],\\n                l2_leaf_reg=spec.get(\'l2\',5), random_seed=seed, thread_count=2,\\n                verbose=False, allow_writing_files=False,\\n                loss_function=f\\"Huber:delta={spec[\'delta\']*1e6:g}\\" if \'delta\' in spec else \'RMSE\')\\n    if spec.get(\'monotone\'):\\n        params[\'monotone_constraints\']={**{c:1 for c in SOCIAL+[\'Hygiene Rating\',\'Restaurant Zomato Rating\']},\'Resturant Tier\':-1}\\n    if spec.get(\'ordered\'): params[\'boosting_type\']=\'Ordered\'\\n    if \'subsample\' in spec: params.update(bootstrap_type=\'Bernoulli\',subsample=spec[\'subsample\'])\\n    if \'random_strength\' in spec: params[\'random_strength\']=spec[\'random_strength\']\\n    fitted=CatBoostRegressor(**params)\\n    target=np.minimum(y,spec[\'cap\']*1e6) if \'cap\' in spec else y\\n    fitted.fit(X,target,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\\n    return fitted\\n\\ndef get_splits(train,scheme):\\n    if str(scheme)==\'city\':\\n        group=features(train,\'engineered\')[\'City\']\\n        splits=list(GroupKFold(5).split(train,groups=group))\\n        for tr,va in splits:assert set(group.iloc[tr]).isdisjoint(group.iloc[va])\\n    else:splits=list(KFold(5,shuffle=True,random_state=int(scheme)).split(train))\\n    counts=np.zeros(len(train),int)\\n    for tr,va in splits:\\n        assert set(tr).isdisjoint(va);counts[va]+=1\\n    assert (counts==1).all()\\n    return splits\\n\\ndef compute_v3(train,y,scheme,contract):\\n    splits=get_splits(train,scheme);pred=np.zeros(len(y))\\n    for name,spec in contract[\'v2_configs\'].items():\\n        if name not in contract[\'v2_weights\']:continue\\n        X=features(train,spec[\'variant\']);component=np.zeros(len(y))\\n        for f,(tr,va) in enumerate(splits):\\n            component[va],_=fit_predict(X,y,tr,va,spec,43+f,spec.get(\'iterations\'))\\n        pred+=.5*contract[\'v2_weights\'][name]*component\\n    X=features(train,\'base\');spec=contract[\'v3_parameters\']\\n    for f,(tr,va) in enumerate(splits):\\n        parts=[]\\n        for seed in contract[\'v3_seeds\']:\\n            model=cat_fit(X.iloc[tr],y[tr],spec,seed+f)\\n            parts.append(np.maximum(model.predict(X.iloc[va]),0))\\n        pred[va]+=.5*np.mean(parts,axis=0)\\n    return pred\\n\\ndef check_submission(frame,test):\\n    assert frame.shape==(500,2),\'Expected 500 rows and two columns\'\\n    assert frame.columns.tolist()==[\'Registration Number\',\'Annual Turnover\'],\'Incorrect header\'\\n    assert frame[\'Registration Number\'].equals(test[\'Registration Number\']),\'Restaurant IDs/order changed\'\\n    assert frame[\'Registration Number\'].is_unique,\'Duplicate restaurant IDs\'\\n    assert np.isfinite(frame[\'Annual Turnover\']).all(),\'Non-finite prediction\'\\n    assert (frame[\'Annual Turnover\']>=0).all(),\'Negative turnover\'\\n\\ndef v5_features(raw,variant=\'base\'):\\n    X=features(raw,\'engineered\' if variant==\'rich\' else \'base\')\\n    if variant in [\'clean\',\'rich\']:\\n        X[\'City\']=X[\'City\'].replace({\'bengaluru\':\'bangalore\',\'banglore\':\'bangalore\',\'gurugram\':\'gurgaon\',\'gurgoan\':\'gurgaon\',\'bombay\':\'mumbai\',\'calcutta\':\'kolkata\',\'delhi\':\'new delhi\',\'bhubneshwar\':\'bhubaneswar\',\'hderabad\':\'hyderabad\',\'punr\':\'pune\'})\\n    if variant==\'rich\':\\n        fb=X[\'Facebook Popularity Quotient\'];ig=X[\'Instagram Popularity Quotient\']\\n        X[\'Social ratio\']=fb/(ig+1)\\n        X[\'Social product\']=fb*ig\\n        X[\'Social minimum\']=np.minimum(fb,ig)\\n        X[\'Social absolute gap\']=np.abs(fb-ig)\\n        X[\'Quality spread\']=X[[\'Food Rating\',\'Hygiene Rating\',\'Overall Restaurant Rating\']].std(axis=1)\\n        X[\'Zomato x Hygiene\']=X[\'Restaurant Zomato Rating\']*X[\'Hygiene Rating\']\\n        X[\'City tier x Restaurant tier\']=X[\'Restaurant City Tier\']*X[\'Resturant Tier\']\\n    return X\\n\\ndef inverse_target(z,power):\\n    if power==0:return np.expm1(np.minimum(z,20))\\n    return np.maximum(z,0)**(1/power)\\n\\ndef fit_v5(raw,y,spec,seed):\\n    X=v5_features(raw,spec.get(\'variant\',\'base\'));target=np.asarray(y,dtype=float)/1e6\\n    if \'cap\' in spec:target=np.minimum(target,spec[\'cap\'])\\n    power=spec.get(\'power\',1.)\\n    transformed=np.log1p(target) if power==0 else target**power\\n    loss=spec.get(\'loss\',\'RMSE\')\\n    params=dict(iterations=spec.get(\'iterations\',850),learning_rate=.04,depth=spec.get(\'depth\',5),\\n                l2_leaf_reg=10,bootstrap_type=\'Bernoulli\',subsample=.7,loss_function=loss,\\n                random_seed=seed,thread_count=2,verbose=False,allow_writing_files=False)\\n    if \'rsm\' in spec:params[\'rsm\']=spec[\'rsm\']\\n    if \'topk\' in spec:\\n        pilot=CatBoostRegressor(**{**params,\'iterations\':300})\\n        pilot.fit(X,transformed,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\\n        ranked=np.argsort(-pilot.feature_importances_);selected=[X.columns[i] for i in ranked[:spec[\'topk\']]]\\n        X=X[selected]\\n    model=CatBoostRegressor(**params)\\n    model.fit(X,transformed,cat_features=X.select_dtypes(include=[\'object\',\'string\']).columns.tolist())\\n    correction=0.\\n    if spec.get(\'correct_mean\'):\\n        # Training-only retransformation correction; all validation labels remain unseen.\\n        fitted_original=inverse_target(model.predict(X),power)\\n        correction=float(np.mean(target-fitted_original))\\n    return {\'model\':model,\'columns\':X.columns.tolist(),\'power\':power,\'correction\':correction,\'spec\':spec}\\n\\ndef predict_v5(fitted,raw):\\n    X=v5_features(raw,fitted[\'spec\'].get(\'variant\',\'base\'))[fitted[\'columns\']]\\n    return np.maximum(0,(inverse_target(fitted[\'model\'].predict(X),fitted[\'power\'])+fitted[\'correction\'])*1e6)\\n\\ndef evaluate(train,y,scheme,contract,previous=None):\\n    if previous is None:previous=compute_v3(train,y,scheme,contract)\\n    prediction=contract[\'v3_weight\']*previous\\n    parts={}\\n    for name,c in contract[\'components\'].items():\\n        p=np.zeros(len(y))\\n        for f,(tr,va) in enumerate(get_splits(train,scheme)):\\n            one=[]\\n            for seed in contract[\'seeds\']:\\n                fitted=fit_v5(train.iloc[tr],y[tr],c[\'spec\'],seed+f)\\n                one.append(predict_v5(fitted,train.iloc[va]))\\n            p[va]=np.mean(one,axis=0)\\n        parts[name]=p;prediction+=c[\'weight\']*p\\n    return {\'previous\':previous,\'candidate\':prediction,**parts}\\n\\ndef full_fit(train,test,y,contract):\\n    prior=np.zeros(len(test));fitted={\'v2\':{},\'v3\':[],\'new\':{}}\\n    tr=np.arange(len(train));va=np.arange(len(train),len(train)+len(test));yy=np.r_[y,np.full(len(test),np.nan)]\\n    for name,w in contract[\'v2_weights\'].items():\\n        spec=contract[\'v2_configs\'][name]\\n        X=pd.concat([features(train,spec[\'variant\']),features(test,spec[\'variant\'])],ignore_index=True)\\n        p,_,model=fit_predict(X,yy,tr,va,spec,42,spec.get(\'iterations\'),return_fitted=True)\\n        fitted[\'v2\'][name]=model;prior+=.5*w*p\\n    for seed in contract[\'v3_seeds\']:\\n        model=cat_fit(features(train,\'base\'),y,contract[\'v3_parameters\'],seed);fitted[\'v3\'].append(model)\\n        prior+=.5*np.maximum(model.predict(features(test,\'base\')),0)/len(contract[\'v3_seeds\'])\\n    candidate=contract[\'v3_weight\']*prior\\n    for name,c in contract[\'components\'].items():\\n        fitted[\'new\'][name]=[]\\n        for seed in contract[\'seeds\']:\\n            model=fit_v5(train,y,c[\'spec\'],seed);fitted[\'new\'][name].append(model)\\n            candidate+=c[\'weight\']*predict_v5(model,test)/len(contract[\'seeds\'])\\n    return prior,candidate,fitted\\n\\ndef predict_final(raw,fitted,contract):\\n    prior=sum(.5*w*predict_fitted(fitted[\'v2\'][name],features(raw,contract[\'v2_configs\'][name][\'variant\'])) for name,w in contract[\'v2_weights\'].items())\\n    prior+=.5*np.mean([np.maximum(m.predict(features(raw,\'base\')),0) for m in fitted[\'v3\']],axis=0)\\n    p=contract[\'v3_weight\']*prior\\n    for name,c in contract[\'components\'].items():p+=c[\'weight\']*np.mean([predict_v5(m,raw) for m in fitted[\'new\'][name]],axis=0)\\n    return p\\n","models.py":"\\"\\"\\"V7: turnover bands, partially pooled tier specialists and nested residual models.\\"\\"\\"\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom catboost import CatBoostClassifier, CatBoostRegressor\\nfrom scipy.special import softmax\\nfrom sklearn.compose import ColumnTransformer\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.linear_model import Ridge\\nfrom sklearn.model_selection import KFold, GroupKFold, StratifiedKFold\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.preprocessing import OneHotEncoder, StandardScaler\\nimport reference as ref\\n\\nREQUIRED = [\'Train_dataset.csv\', \'Test_dataset.csv\', \'Data_Dictionary.csv\', \'Sample_Solution.csv\']\\nSEEDS = [10242, 20242]\\nBANDS = [20., 40., 80.]\\nSPECS = {}\\nfor depth in [3, 5]:\\n    for calibrate in [False, True]:\\n        SPECS[f\'bands_d{depth}_cal{int(calibrate)}\'] = {\\n            \'family\': \'bands\', \'depth\': depth, \'calibrate\': calibrate}\\nfor depth in [4, 6]:\\n    for shrink in [100, 300]:\\n        SPECS[f\'tiers_d{depth}_shrink{shrink}\'] = {\\n            \'family\': \'tiers\', \'depth\': depth, \'shrink\': shrink}\\nfor alpha in [100, 1000]:\\n    SPECS[f\'residual_ridge{alpha}\'] = {\'family\': \'residual\', \'kind\': \'ridge\', \'alpha\': alpha}\\nfor loss in [\'RMSE\', \'Huber:delta=20\']:\\n    SPECS[\'residual_tree_\' + (\'l2\' if loss == \'RMSE\' else \'h20\')] = {\\n        \'family\': \'residual\', \'kind\': \'cat\', \'loss\': loss}\\n\\n\\ndef load_data(directory):\\n    directory = Path(directory)\\n    assert all((directory / p).is_file() for p in REQUIRED), f\'Four official inputs required in {directory}\'\\n    train = pd.read_csv(directory / REQUIRED[0]).rename(columns={\'Endorsed By\': \'Endoresed By\'})\\n    test = pd.read_csv(directory / REQUIRED[1])\\n    sample = pd.read_csv(directory / REQUIRED[3])\\n    pd.read_csv(directory / REQUIRED[2])\\n    assert len(test) == 500 and train[\'Registration Number\'].is_unique and test[\'Registration Number\'].is_unique\\n    assert train.drop(columns=\'Annual Turnover\').columns.tolist() == test.columns.tolist()\\n    assert set(train[\'Registration Number\']).isdisjoint(test[\'Registration Number\'])\\n    assert sample.columns.tolist() == [\'Registration Number\', \'Annual Turnover\']\\n    assert sample[\'Registration Number\'].equals(test[\'Registration Number\'])\\n    y = train[\'Annual Turnover\'].to_numpy(float)\\n    assert np.isfinite(y).all() and (y > 0).all()\\n    return train, test, y\\n\\n\\ndef features(raw):\\n    return ref.v5_features(raw, \'clean\')\\n\\n\\ndef split_indices(raw, scheme):\\n    scheme = str(scheme)\\n    if scheme == \'city\':\\n        groups = features(raw)[\'City\']\\n        result = list(GroupKFold(5).split(raw, groups=groups))\\n        for a, b in result:\\n            assert set(groups.iloc[a]).isdisjoint(groups.iloc[b])\\n    elif scheme.startswith(\'strat:\'):\\n        seed = int(scheme.split(\':\')[1])\\n        bins = np.digitize(raw[\'Annual Turnover\'].to_numpy(float) / 1e6, BANDS, right=True)\\n        result = list(StratifiedKFold(5, shuffle=True, random_state=seed).split(raw, bins))\\n    elif scheme.startswith(\'inner:\'):\\n        seed = int(scheme.split(\':\')[1])\\n        result = list(KFold(3, shuffle=True, random_state=seed).split(raw))\\n    else:\\n        result = list(KFold(5, shuffle=True, random_state=int(scheme)).split(raw))\\n    counts = np.zeros(len(raw), int)\\n    for a, b in result:\\n        assert not set(a).intersection(b)\\n        counts[b] += 1\\n    assert (counts == 1).all()\\n    return result\\n\\n\\ndef cat_params(seed, depth, trees, l2=20):\\n    return dict(iterations=trees, depth=depth, learning_rate=.04, l2_leaf_reg=l2,\\n                bootstrap_type=\'Bernoulli\', subsample=.7, random_seed=seed,\\n                thread_count=2, verbose=False, allow_writing_files=False)\\n\\n\\ndef residual_features(raw, baseline):\\n    X = features(raw)\\n    keep = [\'Facebook Popularity Quotient\', \'Instagram Popularity Quotient\',\\n            \'Resturant Tier\', \'Hygiene Rating\', \'Restaurant Zomato Rating\',\\n            \'Restaurant City Tier\', \'Opening Year\', \'Overall Restaurant Rating\',\\n            \'Cuisine\', \'Restaurant Type\', \'Restaurant Location\']\\n    Z = X[keep].copy()\\n    Z[\'Base prediction\'] = np.asarray(baseline) / 1e6\\n    Z[\'Social mean\'] = Z[[\'Facebook Popularity Quotient\', \'Instagram Popularity Quotient\']].mean(axis=1)\\n    Z[\'Social hygiene\'] = Z[\'Social mean\'] * Z[\'Hygiene Rating\'] / 100\\n    Z[\'Tier1 base\'] = Z[\'Base prediction\'] * (Z[\'Resturant Tier\'] == 1)\\n    Z[\'Hygiene base\'] = Z[\'Base prediction\'] * (Z[\'Hygiene Rating\'] - 5) / 5\\n    return Z\\n\\n\\ndef fit_component(raw, y, spec, seed, inner_baseline=None):\\n    X = features(raw)\\n    target = np.asarray(y, float) / 1e6\\n    cats = X.select_dtypes(include=[\'object\', \'string\']).columns.tolist()\\n    info = {\'seed\': seed, \'train_rows\': len(raw), \'family\': spec[\'family\']}\\n    if spec[\'family\'] == \'bands\':\\n        labels = np.digitize(target, BANDS, right=True)\\n        counts = np.bincount(labels, minlength=4)\\n        assert (counts >= 3).all()\\n        centers = np.array([target[labels == k].mean() for k in range(4)])\\n        temperature = 1.\\n        if spec[\'calibrate\']:\\n            prob = np.zeros((len(X), 4))\\n            for f, (a, b) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed + 411).split(X, labels)):\\n                model = CatBoostClassifier(**cat_params(seed + 100 + f, spec[\'depth\'], 800), loss_function=\'MultiClass\')\\n                model.fit(X.iloc[a], labels[a], cat_features=cats)\\n                assert np.array_equal(model.classes_, np.arange(4))\\n                prob[b] = model.predict_proba(X.iloc[b])\\n            log_prob = np.log(np.clip(prob, 1e-12, 1))\\n            temperatures = [.7, 1., 1.4, 2.]\\n            losses = [-np.mean(np.log(np.clip(softmax(log_prob / t, axis=1)[np.arange(len(X)), labels], 1e-12, 1))) for t in temperatures]\\n            temperature = temperatures[int(np.argmin(losses))]\\n            info.update(temperature_candidates=temperatures, inner_oof_logloss=losses)\\n        model = CatBoostClassifier(**cat_params(seed, spec[\'depth\'], 800), loss_function=\'MultiClass\')\\n        model.fit(X, labels, cat_features=cats)\\n        assert np.array_equal(model.classes_, np.arange(4))\\n        info.update(band_counts=counts.tolist(), band_centers_millions=centers.tolist(), temperature=temperature)\\n        return {\'family\': \'bands\', \'model\': model, \'centers\': centers, \'temperature\': temperature, \'info\': info}\\n    if spec[\'family\'] == \'tiers\':\\n        global_model = CatBoostRegressor(**cat_params(seed, 5, 850, 10), loss_function=\'Huber:delta=40\')\\n        global_model.fit(X, target, cat_features=cats)\\n        local = {}\\n        for tier in [1., 2.]:\\n            mask = X[\'Resturant Tier\'].to_numpy() == tier\\n            n = int(mask.sum())\\n            if n < 120:\\n                continue\\n            model = CatBoostRegressor(**cat_params(seed + int(tier) * 71, spec[\'depth\'], 650, 30), loss_function=\'Huber:delta=40\')\\n            model.fit(X.loc[mask], target[mask], cat_features=cats)\\n            local[tier] = {\'model\': model, \'weight\': n / (n + spec[\'shrink\']), \'rows\': n}\\n        info[\'local_tiers\'] = {str(t): {\'weight\': value[\'weight\'], \'rows\': value[\'rows\']} for t, value in local.items()}\\n        return {\'family\': \'tiers\', \'global\': global_model, \'local\': local, \'info\': info}\\n    assert spec[\'family\'] == \'residual\' and inner_baseline is not None\\n    inner_baseline = np.asarray(inner_baseline, float)\\n    assert inner_baseline.shape == target.shape and np.isfinite(inner_baseline).all()\\n    Z = residual_features(raw, inner_baseline)\\n    residual = target - inner_baseline / 1e6\\n    if spec[\'kind\'] == \'ridge\':\\n        categorical = Z.select_dtypes(include=[\'object\', \'string\']).columns.tolist()\\n        numeric = [c for c in Z if c not in categorical]\\n        pre = ColumnTransformer([\\n            (\'num\', make_pipeline(SimpleImputer(strategy=\'median\', add_indicator=True), StandardScaler()), numeric),\\n            (\'cat\', OneHotEncoder(handle_unknown=\'ignore\', min_frequency=10, sparse_output=False), categorical)])\\n        model = make_pipeline(pre, Ridge(alpha=spec[\'alpha\']))\\n        model.fit(Z, residual)\\n    else:\\n        model = CatBoostRegressor(**cat_params(seed, 2, 300, 50), loss_function=spec[\'loss\'])\\n        model.fit(Z, residual, cat_features=Z.select_dtypes(include=[\'object\', \'string\']).columns.tolist())\\n    info.update(inner_baseline=\'three-fold v5 OOF within this outer training fold\', mean_inner_residual_millions=float(residual.mean()))\\n    return {\'family\': \'residual\', \'model\': model, \'info\': info}\\n\\n\\ndef predict_component(fitted, raw, baseline=None):\\n    if fitted[\'family\'] == \'bands\':\\n        p = fitted[\'model\'].predict_proba(features(raw))\\n        p = softmax(np.log(np.clip(p, 1e-12, 1)) / fitted[\'temperature\'], axis=1)\\n        prediction = p @ fitted[\'centers\'] * 1e6\\n    elif fitted[\'family\'] == \'tiers\':\\n        X = features(raw)\\n        prediction = fitted[\'global\'].predict(X)\\n        for tier, local in fitted[\'local\'].items():\\n            mask = X[\'Resturant Tier\'].to_numpy() == tier\\n            if mask.any():\\n                prediction[mask] = (1 - local[\'weight\']) * prediction[mask] + local[\'weight\'] * local[\'model\'].predict(X.loc[mask])\\n        prediction *= 1e6\\n    else:\\n        assert baseline is not None and len(baseline) == len(raw)\\n        prediction = np.asarray(baseline) + fitted[\'model\'].predict(residual_features(raw, baseline)) * 1e6\\n    prediction = np.maximum(prediction, 0)\\n    assert prediction.shape == (len(raw),) and np.isfinite(prediction).all()\\n    return prediction\\n\\n\\ndef evaluate_component(raw, y, spec, scheme, baseline, inner_provider=None, progress=None):\\n    individual = np.zeros((len(y), len(SEEDS)))\\n    diagnostics = []\\n    for fold, (a, b) in enumerate(split_indices(raw, scheme)):\\n        inner = inner_provider(a, fold) if spec[\'family\'] == \'residual\' else None\\n        for j, seed in enumerate(SEEDS):\\n            fitted = fit_component(raw.iloc[a], y[a], spec, seed + fold, inner)\\n            individual[b, j] = predict_component(fitted, raw.iloc[b], baseline[b])\\n            diagnostics.append({\'fold\': fold, **fitted[\'info\']})\\n        if progress:\\n            progress(f\'fold {fold + 1}/5 complete\')\\n    return individual.mean(axis=1), individual, diagnostics\\n\\n\\ndef v5_fold_prediction(train, y, valid, contract, fold):\\n    \\"\\"\\"Exact legacy outer-fold seed schedule; no validation labels enter fitting.\\"\\"\\"\\n    n = len(train)\\n    a = np.arange(n)\\n    b = np.arange(n, n + len(valid))\\n    joined_y = np.r_[np.asarray(y), np.full(len(valid), np.nan)]\\n    prior = np.zeros(len(valid))\\n    for name, weight in contract[\'v2_weights\'].items():\\n        spec = contract[\'v2_configs\'][name]\\n        X = pd.concat([ref.features(train, spec[\'variant\']), ref.features(valid, spec[\'variant\'])], ignore_index=True)\\n        p, _ = ref.fit_predict(X, joined_y, a, b, spec, 43 + fold, spec.get(\'iterations\'))\\n        prior += .5 * weight * p\\n    for seed in contract[\'v3_seeds\']:\\n        model = ref.cat_fit(ref.features(train, \'base\'), np.asarray(y), contract[\'v3_parameters\'], seed + fold)\\n        prior += .5 * np.maximum(model.predict(ref.features(valid, \'base\')), 0) / len(contract[\'v3_seeds\'])\\n    p = contract[\'v3_weight\'] * prior\\n    for name, component in contract[\'components\'].items():\\n        for seed in contract[\'seeds\']:\\n            model = ref.fit_v5(train, np.asarray(y), component[\'spec\'], seed + fold)\\n            p += component[\'weight\'] * ref.predict_v5(model, valid) / len(contract[\'seeds\'])\\n    return p\\n\\n\\ndef inner_v5_oof(raw, y, contract, seed=731, progress=None):\\n    prediction = np.zeros(len(raw))\\n    for f, (a, b) in enumerate(split_indices(raw, f\'inner:{seed}\')):\\n        prediction[b] = v5_fold_prediction(raw.iloc[a], np.asarray(y)[a], raw.iloc[b], contract, f)\\n        if progress:\\n            progress(f\'inner fold {f + 1}/3 complete\')\\n    return prediction\\n\\n\\ndef fit_submission(train, test, y, contract, choice, inner_baseline=None):\\n    # Fresh baseline, including independent verification against the scored v5 file outside this function.\\n    _, baseline, legacy = ref.full_fit(train, test, y, contract)\\n    fitted = {\'choice\': choice, \'legacy\': legacy, \'contract\': contract, \'models\': []}\\n    prediction = (1 - choice[\'weight\']) * baseline\\n    for seed in SEEDS:\\n        model = fit_component(train, y, SPECS[choice[\'name\']], seed, inner_baseline)\\n        fitted[\'models\'].append(model)\\n        prediction += choice[\'weight\'] * predict_component(model, test, baseline) / len(SEEDS)\\n    return prediction, baseline, fitted\\n\\n\\ndef predict_submission(fitted, raw):\\n    baseline = ref.predict_final(raw, fitted[\'legacy\'], fitted[\'contract\'])\\n    choice = fitted[\'choice\']\\n    component = np.mean([predict_component(m, raw, baseline) for m in fitted[\'models\']], axis=0)\\n    return (1 - choice[\'weight\']) * baseline + choice[\'weight\'] * component\\n","workflow.py":"\\"\\"\\"Resumable bounded v7 research; exports experimental trials without claiming a portal gain.\\"\\"\\"\\nimport argparse\\nimport hashlib\\nimport json\\nimport platform\\nimport time\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nimport catboost, sklearn, scipy\\nfrom sklearn.compose import ColumnTransformer\\nfrom sklearn.pipeline import make_pipeline\\nfrom sklearn.impute import SimpleImputer\\nfrom sklearn.preprocessing import StandardScaler, OneHotEncoder\\nfrom sklearn.neighbors import NearestNeighbors\\nimport reference as ref\\nfrom models import (SPECS, SEEDS, REQUIRED, load_data, features, split_indices,\\n                    fit_component, predict_component, v5_fold_prediction,\\n                    fit_submission, predict_submission)\\n\\nROOT = Path(__file__).resolve().parent\\nBASE = ROOT.parent\\nOUT = ROOT / \'results\'\\nSTEM = \'submission\'\\nCONFIRM = [18421, 18422, \'strat:27421\', \'city\']\\n\\n\\ndef sha(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\n\\ndef digest(value):\\n    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()\\n\\n\\ndef save(path, value):\\n    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\\\n\')\\n\\n\\ndef log(value):\\n    print(value if isinstance(value, str) else json.dumps(value), flush=True)\\n\\n\\ndef safe(value):\\n    return str(value).replace(\':\', \'_\')\\n\\n\\nclass Research:\\n    def __init__(self, data_dir=BASE):\\n        self.train, self.test, self.y = load_data(data_dir)\\n        self.inputs = {name: sha(Path(data_dir) / name) for name in REQUIRED}\\n        self.sources = {name: sha(ROOT / name) for name in [\'models.py\', \'workflow.py\', \'reference.py\', \'scored_reference.json\']}\\n        self.contract = json.loads((ROOT / \'v5_contract.json\').read_text())\\n        OUT.mkdir(exist_ok=True)\\n        (OUT / \'cache\').mkdir(exist_ok=True)\\n\\n    def context(self, name, raw=None, extra=None):\\n        raw = self.train if raw is None else raw\\n        return {\'name\': name, \'inputs\': self.inputs, \'sources\': self.sources,\\n                \'contract\': digest(self.contract), \'spec\': SPECS.get(name), \'seeds\': SEEDS,\\n                \'training_rows\': digest(raw[\'Registration Number\'].tolist()), \'extra\': extra}\\n\\n    def cached(self, name, context, compute):\\n        path = OUT / \'cache\' / (safe(name) + \'.npz\')\\n        meta = path.with_suffix(\'.json\')\\n        if path.exists() and meta.exists():\\n            record = json.loads(meta.read_text())\\n            assert record[\'context\'] == context, f\'Stale checkpoint: {path}\'\\n            assert record[\'sha256\'] == sha(path), f\'Corrupt checkpoint: {path}\'\\n            p = np.load(path, allow_pickle=False)[\'prediction\'].copy()\\n            return p, record.get(\'diagnostics\', {})\\n        p, diagnostics = compute()\\n        p = np.asarray(p, float)\\n        assert np.isfinite(p).all()\\n        # Rename temporary files only after a complete write, to survive interruption.\\n        temporary = path.with_suffix(\'.partial.npz\')\\n        np.savez_compressed(temporary, prediction=p)\\n        temporary.replace(path)\\n        temporary_meta = meta.with_suffix(\'.partial.json\')\\n        save(temporary_meta, {\'context\': context, \'sha256\': sha(path), \'diagnostics\': diagnostics})\\n        temporary_meta.replace(meta)\\n        return p, diagnostics\\n\\n    def audit(self):\\n        old = json.loads((ROOT / \'scored_reference.json\').read_text())\\n        assert old[\'inputs\'] == self.inputs\\n        for ext in [\'csv\', \'ipynb\']:\\n            name = STEM + \'.\' + ext\\n            assert sha(ROOT / \'scored_v5_backup\' / name) == old[\'preserved_v5\'][name]\\n            original = BASE / \'improvement_v5\' / name\\n            if original.exists():\\n                assert sha(original) == old[\'preserved_v5\'][name]\\n        X, T = features(self.train), features(self.test)\\n        changed = self.train.copy()\\n        changed[\'Annual Turnover\'] = 1e30\\n        changed[\'Registration Number\'] = -77\\n        pd.testing.assert_frame_equal(features(changed), X)\\n        assert X.columns.equals(T.columns)\\n        assert np.isclose(ref.rmse([0, 3], [0, 0]), np.sqrt(4.5))\\n        # Local similarity is a diagnostic; it does not establish that a label is wrong.\\n        cats = X.select_dtypes(include=[\'object\', \'string\']).columns.tolist()\\n        nums = [c for c in X if c not in cats]\\n        pre = ColumnTransformer([\\n            (\'num\', make_pipeline(SimpleImputer(strategy=\'median\', add_indicator=True), StandardScaler()), nums),\\n            (\'cat\', OneHotEncoder(handle_unknown=\'ignore\', sparse_output=False), cats)])\\n        transformed = pre.fit_transform(X)\\n        nn = NearestNeighbors(n_neighbors=2, n_jobs=2).fit(transformed)\\n        distances, indices = nn.kneighbors(transformed)\\n        neighbor = self.y[indices[:, 1]]\\n        diagnostic = self.train[[\'Registration Number\', \'Annual Turnover\', \'City\', \'Resturant Tier\',\\n                                 \'Hygiene Rating\', \'Facebook Popularity Quotient\',\\n                                 \'Instagram Popularity Quotient\']].copy()\\n        diagnostic[\'nearest_training_id\'] = self.train[\'Registration Number\'].to_numpy()[indices[:, 1]]\\n        diagnostic[\'nearest_training_turnover\'] = neighbor\\n        diagnostic[\'distance\'] = distances[:, 1]\\n        diagnostic.loc[self.y > 80e6].sort_values(\'Annual Turnover\', ascending=False).to_csv(OUT / \'tail_audit.csv\', index=False)\\n        record = {\'inputs\': self.inputs, \'sources\': self.sources, \'preserved_v5\': old[\'preserved_v5\'],\\n                  \'target_rows\': len(self.y), \'tail_above_80M\': int((self.y > 80e6).sum()),\\n                  \'nearest_neighbor_tail_above80M\': int((neighbor[self.y > 80e6] > 80e6).sum()),\\n                  \'median_nearest_target_gap_INR\': float(np.median(np.abs(self.y - neighbor))),\\n                  \'train_feature_duplicates\': int(X.duplicated().sum()),\\n                  \'cross_train_test_duplicates\': int(pd.concat([X, T]).duplicated().sum() - X.duplicated().sum() - T.duplicated().sum()),\\n                  \'labels_changed\': 0, \'verified_unit_errors\': 0,\\n                  \'label_audit_conclusion\': \'No independently verified unit error; retain all original labels. Dissimilarity is not evidence of an incorrect label.\',\\n                  \'portal_v5_rmse\': 12046722.3691139, \'leader_screenshot_rmse\': 4796740.5600053,\\n                  \'rules_source\': \'full competition instructions supplied by user on 26 October 2026\',\\n                  \'stated_metric\': \'ordinary RMSE across the 500 test predictions\',\\n                  \'csv_submission_limit\': None, \'final_notebook_submissions\': 1,\\n                  \'portal_uploads\': 0, \'external_data_used\': False,\\n                  \'runtime\': {\'python\': platform.python_version(), \'numpy\': np.__version__, \'pandas\': pd.__version__,\\n                              \'catboost\': catboost.__version__, \'sklearn\': sklearn.__version__, \'scipy\': scipy.__version__},\\n                  \'utc\': datetime.now(timezone.utc).isoformat()}\\n        save(OUT / \'audit.json\', record)\\n        plan = {\'specs\': SPECS, \'seeds\': SEEDS, \'development\': [42, 71], \'weights\': [.25, .5, .75, 1.],\\n                \'shortlist\': \'best eligible configuration from each family, development mean <=1% worse, each assignment <=2% worse\',\\n                \'confirmation\': CONFIRM, \'nested_residual_training\': \'three-fold v5 OOF within each outer training fold; checkpoints shared only for identical rows/configurations\',\\n                \'local_champion\': \'positive gain on both random confirmation assignments; city relative increase <=1%\',\\n                \'experimental_trial\': \'random confirmation mean <=1% worse; city <=3% worse; stratified <=2% worse\',\\n                \'export_limit\': 2, \'export_order\': \'local champions first, then mean random confirmation RMSE; distinct families\',\\n                \'scope\': \'original supplied data and original code; no leaderboard-fitted row adjustments\',\\n                \'longer_time\': \'per user authorization; bounded settings, no search expansion after confirmation\'}\\n        save(ROOT / \'experiment_plan.json\', plan)\\n        log({\'audit\': \'PASS\', \'tail\': record[\'tail_above_80M\'], \'labels_changed\': 0, \'settings\': len(SPECS)})\\n\\n    def baseline(self, scheme):\\n        name = f\'v5_{safe(scheme)}\'\\n        context = self.context(\'v5\', extra={\'scheme\': str(scheme), \'folds\': digest([b.tolist() for _, b in split_indices(self.train, scheme)])})\\n\\n        def compute():\\n            if str(scheme) in [\'42\', \'71\', \'18421\'] and (BASE / \'improvement_v6/results\' / f\'v5_{scheme}.npz\').exists():\\n                path = BASE / \'improvement_v6/results\' / f\'v5_{scheme}.npz\'\\n                meta = json.loads(path.with_suffix(\'.json\').read_text())\\n                assert meta[\'context\'][\'inputs\'] == self.inputs\\n                assert meta[\'context\'][\'v5_contract\'] == digest(self.contract)\\n                assert meta[\'context\'][\'fold_sha256\'] == context[\'extra\'][\'folds\']\\n                assert meta[\'sha256\'] == sha(path)\\n                for filename, h in meta[\'context\'][\'sources\'].items():\\n                    assert sha(BASE / \'improvement_v6\' / filename) == h\\n                return np.load(path)[\'oof\'], {\'origin\': \'verified actual v6 frozen-v5 predictions\', \'source_sha256\': sha(path)}\\n            if str(scheme) == \'18422\' and (BASE / \'improvement_v5/results/comparison_18422.npz\').exists():\\n                directory = BASE / \'improvement_v5/results\'\\n                manifest = json.loads((directory / \'cache_manifest.json\').read_text())\\n                path = directory / \'comparison_18422.npz\'\\n                meta = json.loads(path.with_suffix(\'.json\').read_text())\\n                provenance = json.loads((ROOT / \'reference_provenance.json\').read_text())\\n                assert meta[\'context\'][\'inputs\'] == self.inputs == manifest[\'inputs\']\\n                assert meta[\'context\'][\'contract\'] == self.contract\\n                assert meta[\'context\'][\'sources\'] == provenance[\'sources\']\\n                assert sha(path) == meta[\'sha256\'] == manifest[\'files\'][path.name]\\n                return np.load(path)[\'candidate\'], {\'origin\': \'verified actual v5 OOF\', \'source_sha256\': sha(path)}\\n            p = np.zeros(len(self.y))\\n            for f, (a, b) in enumerate(split_indices(self.train, scheme)):\\n                fold_context = self.context(\'v5\', self.train.iloc[a], {\'scheme\': str(scheme), \'fold\': f, \'valid_ids\': self.train.iloc[b][\'Registration Number\'].tolist()})\\n                part, _ = self.cached(f\'v5_{safe(scheme)}_fold{f}\', fold_context,\\n                    lambda: (v5_fold_prediction(self.train.iloc[a], self.y[a], self.train.iloc[b], self.contract, f), {}))\\n                p[b] = part\\n                log(f\'v5 / {scheme}: fold {f + 1}/5 complete\')\\n            return p, {\'origin\': \'fresh frozen-v5 folds on v7 split assignment\'}\\n\\n        prediction, _ = self.cached(name, context, compute)\\n        assert prediction.shape == self.y.shape and (prediction >= 0).all()\\n        return prediction\\n\\n    def inner(self, raw, y, tag, seed=731):\\n        p = np.zeros(len(raw))\\n        for f, (a, b) in enumerate(split_indices(raw, f\'inner:{seed}\')):\\n            context = self.context(\'inner_v5\', raw.iloc[a], {\'tag\': tag, \'fold\': f,\\n                \'valid_ids\': raw.iloc[b][\'Registration Number\'].tolist(), \'seed\': seed,\\n                \'training_target_sha256\': hashlib.sha256(np.asarray(y)[a].tobytes()).hexdigest()})\\n            part, _ = self.cached(f\'inner_{safe(tag)}_fold{f}\', context,\\n                lambda: (v5_fold_prediction(raw.iloc[a], np.asarray(y)[a], raw.iloc[b], self.contract, f), {}))\\n            p[b] = part\\n            log(f\'Nested v5 / {tag}: inner fold {f + 1}/3 complete\')\\n        return p\\n\\n    def candidate(self, name, scheme):\\n        spec = SPECS[name]\\n        baseline = self.baseline(scheme)\\n        parts = np.zeros((len(self.y), len(SEEDS)))\\n        started = time.monotonic()\\n        for f, (a, b) in enumerate(split_indices(self.train, scheme)):\\n            # Cached fold predictions are checked before starting expensive nested fits.\\n            expected = [OUT / \'cache\' / f\'{name}_{safe(scheme)}_fold{f}_seed{seed}.npz\' for seed in SEEDS]\\n            need_fit = not all(path.exists() for path in expected)\\n            inner = self.inner(self.train.iloc[a], self.y[a], f\'{safe(scheme)}_outer{f}\', 731 + f) if spec[\'family\'] == \'residual\' and need_fit else None\\n            for j, seed in enumerate(SEEDS):\\n                context = self.context(name, self.train.iloc[a], {\'scheme\': str(scheme), \'fold\': f,\\n                    \'seed\': seed + f, \'valid_ids\': self.train.iloc[b][\'Registration Number\'].tolist()})\\n\\n                def compute():\\n                    fitted = fit_component(self.train.iloc[a], self.y[a], spec, seed + f, inner)\\n                    return predict_component(fitted, self.train.iloc[b], baseline[b]), fitted[\'info\']\\n\\n                part, _ = self.cached(f\'{name}_{safe(scheme)}_fold{f}_seed{seed}\', context, compute)\\n                parts[b, j] = part\\n            log(f\'{name} / {scheme}: fold {f + 1}/5 complete\')\\n        p = parts.mean(axis=1)\\n        path = OUT / f\'{name}_{safe(scheme)}.npz\'\\n        np.savez_compressed(path, oof=p, individual=parts)\\n        save(path.with_suffix(\'.json\'), {\'context\': self.context(name, extra={\'scheme\': str(scheme)}), \'sha256\': sha(path)})\\n        log({\'name\': name, \'scheme\': str(scheme), \'rmse\': ref.rmse(self.y, p), \'seconds\': time.monotonic() - started})\\n        return p\\n\\n    def screen(self):\\n        baseline = {s: self.baseline(s) for s in [42, 71]}\\n        rows = []\\n        for name, spec in SPECS.items():\\n            predictions = {s: self.candidate(name, s) for s in [42, 71]}\\n            for weight in [.25, .5, .75, 1.]:\\n                row = {\'name\': name, \'family\': spec[\'family\'], \'weight\': weight}\\n                for s in [42, 71]:\\n                    score = ref.rmse(self.y, (1 - weight) * baseline[s] + weight * predictions[s])\\n                    row[f\'rmse_{s}\'] = score\\n                    row[f\'gain_{s}\'] = ref.rmse(self.y, baseline[s]) - score\\n                row[\'mean_rmse\'] = np.mean([row[\'rmse_42\'], row[\'rmse_71\']])\\n                rows.append(row)\\n            pd.DataFrame(rows).sort_values(\'mean_rmse\').to_csv(OUT / \'screening.csv\', index=False)\\n        table = pd.DataFrame(rows).sort_values([\'mean_rmse\', \'name\', \'weight\'])\\n        b42, b71 = [ref.rmse(self.y, baseline[s]) for s in [42, 71]]\\n        eligible = table[(table.mean_rmse <= 1.01 * np.mean([b42, b71])) &\\n                         (table.rmse_42 <= 1.02 * b42) & (table.rmse_71 <= 1.02 * b71)]\\n        shortlist = eligible.groupby(\'family\', sort=False).head(1).to_dict(orient=\'records\')\\n        lock = {\'shortlist\': shortlist, \'inputs\': self.inputs, \'sources\': self.sources,\\n                \'specs\': SPECS, \'frozen_before_new_confirmation\': True,\\n                \'utc\': datetime.now(timezone.utc).isoformat()}\\n        path = OUT / \'candidate_lock.json\'\\n        if path.exists():\\n            saved = json.loads(path.read_text())\\n            assert saved[\'shortlist\'] == shortlist and saved[\'sources\'] == self.sources and saved[\'inputs\'] == self.inputs\\n        else:\\n            save(path, lock)\\n        log({\'FROZEN_SHORTLIST\': shortlist, \'sha256\': sha(path)})\\n\\n    def confirm(self):\\n        lock = json.loads((OUT / \'candidate_lock.json\').read_text())\\n        assert lock[\'inputs\'] == self.inputs and lock[\'sources\'] == self.sources\\n        rows, segments, folds, decisions = [], [], [], []\\n        for choice in lock[\'shortlist\']:\\n            metrics = {}\\n            for scheme in CONFIRM:\\n                b = self.baseline(scheme)\\n                component = self.candidate(choice[\'name\'], scheme)\\n                p = (1 - choice[\'weight\']) * b + choice[\'weight\'] * component\\n                old, new = ref.rmse(self.y, b), ref.rmse(self.y, p)\\n                metrics[str(scheme)] = {\'baseline\': old, \'candidate\': new, \'gain\': old - new, \'relative_change\': new / old - 1}\\n                rows.append({\'name\': choice[\'name\'], \'family\': choice[\'family\'], \'weight\': choice[\'weight\'],\\n                             \'scheme\': str(scheme), \'v5_rmse\': old, \'candidate_rmse\': new, \'gain\': old - new})\\n                for label, mask in [(\'<=40M\', self.y <= 40e6), (\'40-80M\', (self.y > 40e6) & (self.y <= 80e6)), (\'>80M\', self.y > 80e6)]:\\n                    segments.append({\'name\': choice[\'name\'], \'scheme\': str(scheme), \'segment\': label,\\n                        \'rows\': int(mask.sum()), \'v5_rmse\': ref.rmse(self.y[mask], b[mask]),\\n                        \'candidate_rmse\': ref.rmse(self.y[mask], p[mask]),\\n                        \'candidate_sse_share\': float(((self.y[mask] - p[mask]) ** 2).sum() / ((self.y - p) ** 2).sum())})\\n                for f, (_, indices) in enumerate(split_indices(self.train, scheme)):\\n                    folds.append({\'name\': choice[\'name\'], \'scheme\': str(scheme), \'fold\': f,\\n                        \'rows\': len(indices), \'tail_rows\': int((self.y[indices] > 80e6).sum()),\\n                        \'v5_rmse\': ref.rmse(self.y[indices], b[indices]),\\n                        \'candidate_rmse\': ref.rmse(self.y[indices], p[indices])})\\n                pd.DataFrame(rows).to_csv(OUT / \'confirmation.csv\', index=False)\\n                pd.DataFrame(segments).to_csv(OUT / \'segments.csv\', index=False)\\n                pd.DataFrame(folds).to_csv(OUT / \'folds.csv\', index=False)\\n            gain = np.mean([metrics[s][\'gain\'] for s in [\'18421\', \'18422\']])\\n            baseline_mean = np.mean([metrics[s][\'baseline\'] for s in [\'18421\', \'18422\']])\\n            champion = all(metrics[s][\'gain\'] > 0 for s in [\'18421\', \'18422\']) and metrics[\'city\'][\'relative_change\'] <= .01\\n            exploratory = (-gain / baseline_mean <= .01 and metrics[\'city\'][\'relative_change\'] <= .03 and\\n                           metrics[\'strat:27421\'][\'relative_change\'] <= .02)\\n            decisions.append({**choice, \'metrics\': metrics, \'local_champion\': bool(champion),\\n                \'experimental_eligible\': bool(exploratory), \'mean_confirmation_gain\': float(gain)})\\n        qualified = sorted([d for d in decisions if d[\'local_champion\'] or d[\'experimental_eligible\']],\\n                           key=lambda d: (not d[\'local_champion\'], -d[\'mean_confirmation_gain\']))[:2]\\n        save(OUT / \'decision.json\', {\'candidates\': decisions, \'trials\': qualified,\\n            \'baseline_portal_rmse\': 12046722.3691139, \'lock_sha256\': sha(OUT / \'candidate_lock.json\'),\\n            \'decision\': \'experimental CSVs for portal evaluation; keep best observed score\' if qualified else \'no stable experimental candidate; preserve v5\',\\n            \'limitation\': \'All labels have been examined previously. No portal improvement is claimed.\'})\\n        log({\'DECISION\': json.loads((OUT / \'decision.json\').read_text())})\\n\\n    def fit(self):\\n        decision = json.loads((OUT / \'decision.json\').read_text())\\n        checks = []\\n        for index, choice in enumerate(decision[\'trials\'], 1):\\n            inner = self.inner(self.train, self.y, \'full_training\', 731) if choice[\'family\'] == \'residual\' else None\\n            log({\'fresh_full_fit\': index, \'choice\': choice[\'name\'], \'weight\': choice[\'weight\']})\\n            p, b, fitted = fit_submission(self.train, self.test, self.y, self.contract, choice, inner)\\n            scored = pd.read_csv(ROOT / \'scored_v5_backup\' / f\'{STEM}.csv\')[\'Annual Turnover\'].to_numpy()\\n            assert np.max(np.abs(b - scored)) < .01\\n            verified = check_behavior(fitted, self.test, p)\\n            directory = ROOT / \'candidates\' / f\'trial_{index}\'\\n            directory.mkdir(parents=True, exist_ok=True)\\n            path = directory / f\'{STEM}.csv\'\\n            pd.DataFrame({\'Registration Number\': self.test[\'Registration Number\'], \'Annual Turnover\': p}).to_csv(path, index=False)\\n            exported = pd.read_csv(path)\\n            ref.check_submission(exported, self.test)\\n            assert np.max(np.abs(exported[\'Annual Turnover\'].to_numpy() - p)) < .01\\n            np.savez_compressed(OUT / f\'full_fit_trial_{index}.npz\', prediction=p, baseline=b)\\n            difference = float(np.sqrt(np.mean((p - scored) ** 2)))\\n            verified.update(trial=f\'trial_{index}\', choice=choice, csv_sha256=sha(path),\\n                prediction_rms_difference_from_v5=difference,\\n                conditional_portal_rmse_lower_bound=max(0., 12046722.3691139 - difference),\\n                can_mathematically_beat_shown_leader=bool(difference > 12046722.3691139 - 4796740.5600053),\\n                bound_assumptions=\'same 500 rows, given v5 score, ordinary unweighted RMSE as stated in rules\',\\n                model_diagnostics=[m[\'info\'] for m in fitted[\'models\']])\\n            checks.append(verified)\\n            save(OUT / \'submission_checks.json\', checks)\\n            log({\'exported\': str(path), \'rms_difference\': difference, \'lower_bound\': verified[\'conditional_portal_rmse_lower_bound\']})\\n\\n\\ndef check_behavior(fitted, test, prediction):\\n    permutation = np.random.default_rng(91).permutation(len(test))\\n    assert np.max(np.abs(predict_submission(fitted, test.iloc[permutation]) - prediction[permutation])) < .01\\n    edge = test.iloc[:3].copy()\\n    edge[\'City\'] = \'unseen_city_v7\'\\n    edge[\'Cuisine\'] = pd.NA\\n    edge[\'Facebook Popularity Quotient\'] = np.nan\\n    edge[\'Resturant Tier\'] = np.nan\\n    assert np.isfinite(predict_submission(fitted, edge)).all()\\n    changed = test.copy()\\n    changed[\'Registration Number\'] = -777\\n    changed[\'Annual Turnover\'] = 1e30\\n    assert np.max(np.abs(predict_submission(fitted, changed) - prediction)) < .01\\n    frame = pd.DataFrame({\'Registration Number\': test[\'Registration Number\'], \'Annual Turnover\': prediction})\\n    ref.check_submission(frame, test)\\n    invalid = [frame.iloc[:-1], pd.concat([frame, frame.iloc[:1]], ignore_index=True),\\n               frame.iloc[::-1].reset_index(drop=True), frame.iloc[:, ::-1],\\n               frame.rename(columns={\'Annual Turnover\': \'Turnover\'}), frame.assign(index=0)]\\n    for column, value in [(\'Registration Number\', \'submission\'), (\'Registration Number\', test[\'Registration Number\'].iloc[1]),\\n                           (\'Annual Turnover\', np.nan), (\'Annual Turnover\', np.inf), (\'Annual Turnover\', -np.inf), (\'Annual Turnover\', -1.)]:\\n        bad = frame.copy()\\n        if isinstance(value, str):\\n            bad[column] = bad[column].astype(object)\\n        bad.loc[0, column] = value\\n        invalid.append(bad)\\n    for bad in invalid:\\n        try:\\n            ref.check_submission(bad, test)\\n        except (AssertionError, ValueError, TypeError):\\n            continue\\n        raise AssertionError(\'Malformed submission accepted\')\\n    return {\'rows\': len(test), \'columns\': frame.columns.tolist(), \'finite_nonnegative\': True,\\n            \'row_order\': True, \'unique_ids\': True, \'permutation_invariance\': True,\\n            \'target_id_invariance\': True, \'missing_unseen_values\': True,\\n            \'malformed_cases_rejected\': len(invalid)}\\n\\n\\nif __name__ == \'__main__\':\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'stage\', choices=[\'audit\', \'screen\', \'confirm\', \'fit\', \'all\'])\\n    parser.add_argument(\'--data-dir\', type=Path, default=BASE)\\n    args = parser.parse_args()\\n    run = Research(args.data_dir)\\n    for stage in ([\'audit\', \'screen\', \'confirm\', \'fit\'] if args.stage == \'all\' else [args.stage]):\\n        getattr(run, stage)()\\n","prepare_resume.py":"\\"\\"\\"Quarantine orphaned research checkpoints before starting a stopped workflow.\\"\\"\\"\\nimport argparse\\nfrom pathlib import Path\\n\\n\\ndef prepare(root):\\n    cache = Path(root) / \'results/cache\'\\n    recovered = []\\n    if cache.exists():\\n        for path in cache.glob(\'*.npz\'):\\n            if \'.incomplete\' in path.stem:\\n                continue\\n            if not path.with_suffix(\'.json\').exists():\\n                destination = path.with_suffix(\'.incomplete.npz\')\\n                number = 1\\n                while destination.exists():\\n                    destination = path.with_suffix(f\'.incomplete{number}.npz\')\\n                    number += 1\\n                path.rename(destination)\\n                recovered.append(str(destination))\\n    return recovered\\n\\n\\nif __name__ == \'__main__\':\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--root\', type=Path, default=Path(__file__).resolve().parent)\\n    args = parser.parse_args()\\n    print(\'Orphan checkpoints preserved for recovery:\', prepare(args.root))\\n","v5_contract.json":"{\\n  \\"v2_configs\\": {\\n    \\"engineered\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"engineered\\",\\n      \\"depth\\": 6,\\n      \\"iterations\\": 260\\n    },\\n    \\"engineered_drop\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"engineered_drop\\",\\n      \\"depth\\": 6,\\n      \\"iterations\\": 216\\n    },\\n    \\"ordered_depth5\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"base\\",\\n      \\"depth\\": 5,\\n      \\"boosting_type\\": \\"Ordered\\",\\n      \\"iterations\\": 219\\n    },\\n    \\"huber20_engineered\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"engineered_drop\\",\\n      \\"depth\\": 5,\\n      \\"loss\\": \\"Huber:delta=20000000\\",\\n      \\"iterations\\": 699\\n    },\\n    \\"huber40\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"base\\",\\n      \\"depth\\": 5,\\n      \\"loss\\": \\"Huber:delta=40000000\\",\\n      \\"iterations\\": 664\\n    },\\n    \\"clip120\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"base\\",\\n      \\"depth\\": 5,\\n      \\"cap\\": 120000000.0,\\n      \\"iterations\\": 591\\n    },\\n    \\"spline_ridge\\": {\\n      \\"kind\\": \\"spline\\",\\n      \\"variant\\": \\"base\\",\\n      \\"alpha\\": 100\\n    },\\n    \\"svr_core\\": {\\n      \\"kind\\": \\"svr\\",\\n      \\"variant\\": \\"core\\",\\n      \\"C\\": 20\\n    },\\n    \\"huber40_engineered\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"engineered\\",\\n      \\"depth\\": 5,\\n      \\"loss\\": \\"Huber:delta=40000000\\",\\n      \\"iterations\\": 619\\n    },\\n    \\"huber40_monotone\\": {\\n      \\"kind\\": \\"cat\\",\\n      \\"variant\\": \\"base\\",\\n      \\"depth\\": 5,\\n      \\"loss\\": \\"Huber:delta=40000000\\",\\n      \\"monotone\\": true,\\n      \\"iterations\\": 447\\n    }\\n  },\\n  \\"v2_weights\\": {\\n    \\"engineered\\": 0.06448907249408362,\\n    \\"engineered_drop\\": 0.09631834163153197,\\n    \\"ordered_depth5\\": 0.08507039177778712,\\n    \\"huber20_engineered\\": 0.06274452861282048,\\n    \\"huber40\\": 0.08855362607985372,\\n    \\"clip120\\": 0.07794631558814363,\\n    \\"spline_ridge\\": 0.15648185431084338,\\n    \\"svr_core\\": 0.058428294374292714,\\n    \\"huber40_engineered\\": 0.09327747966602483,\\n    \\"huber40_monotone\\": 0.2166900954646185\\n  },\\n  \\"v3_parameters\\": {\\n    \\"variant\\": \\"base\\",\\n    \\"depth\\": 5,\\n    \\"iterations\\": 850,\\n    \\"delta\\": 40,\\n    \\"subsample\\": 0.7,\\n    \\"l2\\": 10\\n  },\\n  \\"v3_seeds\\": [\\n    1042,\\n    2042,\\n    3042\\n  ],\\n  \\"version\\": \\"v5\\",\\n  \\"v3_weight\\": 0.25,\\n  \\"seeds\\": [\\n    10242,\\n    20242\\n  ],\\n  \\"components\\": {\\n    \\"sqrt_d6_correct1\\": {\\n      \\"spec\\": {\\n        \\"power\\": 0.5,\\n        \\"depth\\": 6,\\n        \\"correct_mean\\": true\\n      },\\n      \\"weight\\": 0.25\\n    },\\n    \\"subspace_h40\\": {\\n      \\"spec\\": {\\n        \\"loss\\": \\"Huber:delta=40\\",\\n        \\"rsm\\": 0.5,\\n        \\"iterations\\": 1200\\n      },\\n      \\"weight\\": 0.25\\n    },\\n    \\"quantile0.6\\": {\\n      \\"spec\\": {\\n        \\"loss\\": \\"Quantile:alpha=0.6\\",\\n        \\"depth\\": 5\\n      },\\n      \\"weight\\": 0.25\\n    }\\n  },\\n  \\"development\\": {\\n    \\"settings\\": 22,\\n    \\"shortlisted_settings\\": 6,\\n    \\"splits\\": [\\n      42,\\n      71\\n    ],\\n    \\"selection\\": {\\n      \\"name\\": \\"top3\\",\\n      \\"new_weight\\": 0.75,\\n      \\"rmse42\\": 19817236.327581998,\\n      \\"rmse71\\": 19840250.361659143,\\n      \\"mean_rmse\\": 19828743.34462057,\\n      \\"gain42\\": 21791.213259298354,\\n      \\"gain71\\": 29833.246005315334\\n    },\\n    \\"screening_is_optimistic\\": true\\n  },\\n  \\"confirmation\\": {\\n    \\"new_splits\\": [\\n      18421,\\n      18422\\n    ],\\n    \\"previous_splits\\": [\\n      2026,\\n      303\\n    ],\\n    \\"city_stress\\": true,\\n    \\"all_splits_reuse_previously_seen_labels\\": true\\n  }\\n}\\n","reference_provenance.json":"{\\n  \\"method\\": \\"AST extraction of exact function source; no old module imports\\",\\n  \\"sources\\": {\\n    \\"improvement_v2/experiment.py\\": \\"48d0a81875b5814239f244621c0af0d8a5b0aea03815b6e91fdc79ec53a28fed\\",\\n    \\"improvement_v3/experiments.py\\": \\"cc5d39f95855f05bea25340a2fadf23ecf0941aa23c9a453a9c45eb315df4693\\",\\n    \\"improvement_v4/pipeline.py\\": \\"c10b9cb16d13acbdfa0ff91a5f7bad3fe633d2145e26d61f5d9638a769b8e93c\\",\\n    \\"improvement_v5/new_models.py\\": \\"f4f3949193dcc89d8917a93b4aeac9c8643e9e109739d87d5445efdb35bdd223\\",\\n    \\"improvement_v5/pipeline.py\\": \\"0ae5f107133da239b9bd6ae1a886a44be921c6da2a117405a0fcc3e4ac18d0ea\\"\\n  },\\n  \\"reference_sha256\\": \\"1a24f9f116170ea54bd3242264d5cd2a150c3429e039a50cb11c3112e52247c1\\"\\n}\\n","scored_reference.json":"{\\n  \\"inputs\\": {\\n    \\"Train_dataset.csv\\": \\"ec40ebda74ba0bfbfe1af4417a4266a70eadf6e595500e99e5435e7bdc7ee51f\\",\\n    \\"Test_dataset.csv\\": \\"ee4137566adb1c0ce356a8c370e7b8a22f681bde492068efe11346d8d44b4f4d\\",\\n    \\"Data_Dictionary.csv\\": \\"2d86c00e27448593564a161197b6639ee1eb90c01fa523b67e7a09d2ef405c5e\\",\\n    \\"Sample_Solution.csv\\": \\"9519ba4d51eb2d59b8a71b054a62e1f4d48b72cf92930c31f5e5393a2cd135fb\\"\\n  },\\n  \\"preserved_v5\\": {\\n    \\"submission.csv\\": \\"66a673efc5f31bc9f4010de033999fd3712444b8f6dd0478c97e77bb3934630d\\",\\n    \\"submission.ipynb\\": \\"46b7366227f295844988b8512d012ecfa141aa7b81b42df42f565082e4880f48\\"\\n  }\\n}\\n"}')
PRESERVED_V5 = json.loads('{"submission.ipynb":"...","submission.csv":"UmVnaXN0cmF0aW9uIE51bWJlcixBbm51YWwgVHVybm92ZXIKMjAwMDEsMzA2MzI3ODEuNDA4NjYzNTkzCjIwMDAyLDM5NjQwNDEzLjkxMjI2MzA1CjIwMDAzLDI4NjI2NTkxLjg5OTM0NzcyMwoyMDAwNCw0Mjk5MzMzMC4zMzQyNDAyMwoyMDAwNSw0NjI3NjE5Mi41NzM1MDQ2OQoyMDAwNiwzNzQ2NzQ0MS4xNTMwNTkyMgoyMDAwNywzMzQxNzE2OS44MjYyNDY1OQoyMDAwOCwzMjg4MTIxOC43MzMzNjQzNTUKMjAwMDksMjk0MjAzMDEuOTE5MzA5NTY0CjIwMDEwLDI1NTE2NjUwLjQzOTk2NDI3NgoyMDAxMSwyODQwMTcyOS40Mjc5NzI5MTMKMjAwMTIsMzQwNjQ2MjkuMTQ0OTYxNTcKMjAwMTMsMzU3MDI2OTIuOTQ5MzI4MDQKMjAwMTQsMjI4NTIwMzUuODc2MDk3MDUzCjIwMDE1LDE5ODQzNDU4LjgzMzY5ODY4NgoyMDAxNiw0MTAzMDUwNC41NTMxMTk4NAoyMDAxNywzOTM5MTU2MC4wMjgwNDU4MQoyMDAxOCwxNDY1Njk0My4yNjg3MjM5OTYKMjAwMTksMzYyMjU5ODkuMjA5NjY2NDkKMjAwMjAsMjg4NTY0ODYuNjc2NzgyODI0CjIwMDIxLDE3NDMyMzYwLjQ3NzQ2MDU0NQoyMDAyMiwzMDgzNTU4Ni41NjQxMDk2OQoyMDAyMywyNDg5NTk2NS43Mzg0NDc0OAoyMDAyNCwyMjc3MTI2NC41ODkwODI3MDMKMjAwMjUsMzkwOTA1ODUuMjgwNjk4NwoyMDAyNiwxNTc3MTkzNy40NDcyMjI2OAoyMDAyNywzODIxOTM2Ny43NTE2NTcyMgoyMDAyOCwzNjc3MTE5OC4xOTYxMTg1NTYKMjAwMjksMzA4OTQ2NTMuODIzNDM3MzU1CjIwMDMwLDIxMjU1NzU0LjQ1NjU4NzI2MgoyMDAzMSw0MjExODA4Ny42NTU5Njg5OTQKMjAwMzIsMzgwNTUzODUuMzI5NzQ4ODU0CjIwMDMzLDI3OTc3NDM5LjUyMDY2NzYwNQoyMDAzNCwzNjk1NDc4NS41NjU5Mjk2NAoyMDAzNSwyMTcwMjg0OS43NDI3Njc4NgoyMDAzNiwzMDU5MjIxNy43Nzk1MTQxNgoyMDAzNyw0MDU3NTM5MS4wOTYzMjMxNgoyMDAzOCwzMzk5Mzk3OS44NjYzNjI4MjUKMjAwMzksMjExMjg0OTAuNDQyMjQ4MjY2CjIwMDQwLDMyNzUwOTM0LjkyNDIzMTA0NQoyMDA0MSwyODIwNzgzNi4zNjkzMzA3ODYKMjAwNDIsMjgzNzA5NDYuOTQyMjc4ODM2CjIwMDQzLDI2NTM5NTEwLjM2NDAwMjgzNQoyMDA0NCwyNjMyMzg5MS40MTMzMzYxNwoyMDA0NSwzOTI4NDU4Mi42MTQ0NDE2OAoyMDA0NiwxOTA1Nzc4Ni43Mzk2NTI4ODcKMjAwNDcsMjY3MDc1MTAuNjcyODUxODQKMjAwNDgsMzc1MjYzOTguNzk2MTQxMDcKMjAwNDksMjQxODQ0OTcuNjE3NzY4NDY2CjIwMDUwLDMxMTU1ODY0Ljg3MTk2NTIyMgoyMDA1MSwzOTg4MzExNi40Mzk3OTU5NAoyMDA1Miw0MzczODAyOS41Mjk0MjM3NwoyMDA1MywyNDcxODIzNC4zMDQwOTk5NzMKMjAwNTQsMjY2ODczMTAuMDYzMTU4OTQ4CjIwMDU1LDM5MjU5NDc1LjMzNDIxMDUxCjIwMDU2LDMyODgwOTk5LjU2NjE0NzA3NAoyMDA1NywyMDk5Mzg1NS4xNTA1MzI0OTUKMjAwNTgsMzIyNjMzNzMuMzg3NjQxNjE2CjIwMDU5LDMyOTI1NTA1LjUwODExNDI1CjIwMDYwLDI5OTMyNzI4LjAxMzE0OTA2OAoyMDA2MSwxODU0OTA3Ny40MTc2NDk5MDYKMjAwNjIsMzE2MTI5MjMuNjU3ODA0NjUzCjIwMDYzLDMzMDczODM0LjY2MDY1NzgxMgoyMDA2NCw0NzUwMzQxMS4yMDk2NjY5NAoyMDA2NSwzMTU1ODIzOS42MTAwMzM1ODcKMjAwNjYsMzUyMDQzMTAuNzU2NDI1ODEKMjAwNjcsMjAwMjYzOTAuMDc0Njk4MDQKMjAwNjgsMTQ2MDQ4MDQuMjMwMTA2OTgzCjIwMDY5LDI2MTk2ODY1LjQ2NTkwNDY5OAoyMDA3MCw0NTU3NTMxMS4yNTMxMTg2OTQKMjAwNzEsMjM2MDgzMjIuOTQ5ODg2NzgKMjAwNzIsMzE0NDczMTMuNTAzNzExMDEKMjAwNzMsMjU3MzE3MzguMjk0NzA5NTc0CjIwMDc0LDIyMTI1MDM4LjQzNjA3MzUwOAoyMDA3NSwyNjcyMjYzMy4xNzMwNTUxODMKMjAwNzYsMzA2MzM2NDEuMjI2MDk5MTcKMjAwNzcsMzE1MTE2OTYuMjI4MjMxNzk1CjIwMDc4LDMzMzU4MzIyLjgzODkyNTEzOAoyMDA3OSwzMDk5MTc2Ni44NzIwNTQyMwoyMDA4MCwyNTUwMDA1Mi44OTU3NTk5MgoyMDA4MSwzMzg3MjQzNS41MDI0NzM4MjQKMjAwODIsMzQyODMxNTEuMTc5MzcwNjIKMjAwODMsMjgwODkyNzAuMTU3NDE1MTM3CjIwMDg0LDM2NDIxOTc3LjgxMjU0NTQ1CjIwMDg1LDI5ODM5MDg2Ljk4MDYzMTA4CjIwMDg2LDMwMjE0NTc5LjM0MjQ5MDYwNgoyMDA4NywyNjQ3MzY4MC41MDU5OTc4MjUKMjAwODgsMzYwNDgyMTEuODI5MDgyNzYKMjAwODksMzg3NzA3MDYuNzI0MDM0ODQ2CjIwMDkwLDMyMjM0NjI5Ljc2NjU2NTk0CjIwMDkxLDI4OTUxMjQ4LjcxMTMwNjk2CjIwMDkyLDE1OTk3MDQ3LjIxNzExODQ0NAoyMDA5MywyNTEwNTMyMS45MTY5MzgwNTUKMjAwOTQsMTY1MTkzODMuOTIyMjQ3Njk5CjIwMDk1LDM3Nzg3NzM1LjA5Mzc2MzEyCjIwMDk2LDMwNTMyNTc1LjYwMzQ4NDY4MwoyMDA5NywyNzY1MDk4My41MTQxMzU2CjIwMDk4LDI2NTc5NTA2Ljg0NTY5MTg5MwoyMDA5OSwzMzEwMTA5Mi4xMDc4NTU5NTMKMjAxMDAsMjQ1MzY3NzIuNjQ3MzUyMjE1CjIwMTAxLDMwOTQ0MTc3Ljc1NTc5NTgzMwoyMDEwMiwzNTQyMTk1Ni42NDgyODAzNAoyMDEwMywzNTM1NTAwNS44MTgwNzgzNQoyMDEwNCwyMzU4NDYxNS43MjA2MDM4NzYKMjAxMDUsNDA2Njg2MDIuMDc3NTg0NjIKMjAxMDYsNDI2NTQ3OTguOTU3MTg3MTMKMjAxMDcsMjgwMTI0MTcuODY4MjYyNTYzCjIwMTA4LDMxMjY0OTM2LjEwOTgzMjE3NQoyMDEwOSwzMTYyMDY1Ni4zMTA1Njk0NjUKMjAxMTAsMjAzMjI1NDIuNzA3ODMwMjEzCjIwMTExLDM1NDg1MTYzLjYwNDU0OTkzCjIwMTEyLDM4MjUyMDY1Ljk1MTcwNjkKMjAxMTMsMjIwNjQyMTAuNjAyMzAzOTg2CjIwMTE0LDM3ODMxNTcxLjc2ODUzNjAxCjIwMTE1LDM3Mjg3ODIwLjE3MDAyMjcyNgoyMDExNiw0NjMwOTUzMi4wOTUzNDk5NQoyMDExNywxODI5Mjg3NC4xNTQ0MzU4NzMKMjAxMTgsNDA4MjUyMzIuNTY4MDc5MzEKMjAxMTksMjI3MDU5NDEuNDM2NjY5MwoyMDEyMCw0NzMyMjc5MC42NDY2NDcwOTYKMjAxMjEsMTgzOTkwNDAuODA2NTY0MDE4CjIwMTIyLDMxMTcyNTc1LjI4NzU0OTE5CjIwMTIzLDI5OTQ4NDQzLjcxMTc5OTg4NgoyMDEyNCwxNjg2MjEwOC4yMjQ3MTE5MwoyMDEyNSw0OTkxODk2My4yMzgwODk3MwoyMDEyNiwyNzI4MTc0NS44NTc4ODczODcKMjAxMjcsMjQzMDc0OTQuNzA1MDQ4MDYKMjAxMjgsMTkxNDAxNTYuMjY1MDE5MTUKMjAxMjksMzMxMzQ0ODkuNzA1ODMzOTY4CjIwMTMwLDM4MzYyMTQ0LjI4MDUwNjE1CjIwMTMxLDIyNjgxOTY1LjA5NDI1OTI5CjIwMTMyLDIzNTQ5Nzg5Ljc5MTM2NTQ1NgoyMDEzMywyNzk0Nzc2My4xNDUxMDkxMwoyMDEzNCwzNTE3MzMwMS45NzUxMDgyMDYKMjAxMzUsMzE5NTkzNzYuNDg3MjY3NzA3CjIwMTM2LDI0MTY4ODExLjYwMDAxOTc2OAoyMDEzNywzNTYwMTc2OC4yMjY0MTkyMwoyMDEzOCwzMDE5MzUzOC42MjMyNTkzNAoyMDEzOSwzNTcxMDAyMC40Mzg4NzEyMzUKMjAxNDAsMjczMDM0NzguMjY3NTI0NDYKMjAxNDEsMzY3MjA2NjIuNzkwNDY1NjEKMjAxNDIsMzEwNTQ4NzUuMDY2NTUyMjgKMjAxNDMsMzMwNzM3MTYuMjM0MzYxNjI2CjIwMTQ0LDIxNDA3MzI0LjUwODk0NDM5NgoyMDE0NSwyNjIwODU5Ny4zNjk3ODg2NTgKMjAxNDYsMjk4NTU0ODYuMTc0OTIwNjQ1CjIwMTQ3LDQzNzUzMTIwLjQyMzI1ODYyNQoyMDE0OCwyOTg5NDU1MC43NjYzMTk3NDgKMjAxNDksMzA1NjEyNDYuNDQ0NDYzOTk0CjIwMTUwLDI3NDMwODY1LjgzNTU0NzMxMwoyMDE1MSwzNzYxNTQ0Ni4wNDY2MDY5OAoyMDE1MiwxOTEyOTg0MC42MzQwMjA5MzYKMjAxNTMsMjAxNzIxNjAuMDU3NTc0MjcyCjIwMTU0LDI4ODkxMTAwLjEzMzA1OTI0CjIwMTU1LDM1NzM2NDE0LjA1NDI1MDkyNgoyMDE1NiwxNzQ5MzY3My4zNjM4MjA0ODYKMjAxNTcsMzAwMzg4OTEuNDU2NDA5ODIzCjIwMTU4LDM3MTI0MTE0LjUzNDM3MzI2CjIwMTU5LDI3MjUyMDIwLjAwNTUzNDIwNgoyMDE2MCwzMjc1ODkzMi44ODY5NjIwOTMKMjAxNjEsMzMxOTAyNDUuMTY4OTk4NzEKMjAxNjIsMjcyNjY2ODYuNTMwMzMzMDUKMjAxNjMsMzg5NDEyNDMuMDY1MzQ1MDQKMjAxNjQsNDcxNTY4NzQuMzMwMTU5NjA1CjIwMTY1LDM0MzIxNjY2LjYwMzI5OTc2CjIwMTY2LDM4NjAxMTgwLjYzMDk5MjM3NQoyMDE2NywyNTU2OTM2NC43ODQwMTEzNTcKMjAxNjgsMTc5MzU0NjAuMzczOTk1NzEKMjAxNjksMzg3NDY4MTQuNDE0NzA3NTQKMjAxNzAsMjYyMjUyNDYuMDg1NjIyOQoyMDE3MSw0NTk2NDc1NC44MjU0OTAwOQoyMDE3MiwzMzg4MzQ5MS4yMTg1NTQwMgoyMDE3MywyNDA4MTc3Ny41NTkxNTExOTUKMjAxNzQsMzUwNDg5ODIuOTAxNTkxODUKMjAxNzUsMjg0MTM1NDguMDA5NTE2MzMKMjAxNzYsMzUzMzY0ODkuMDIzMDEzMwoyMDE3NywzMTM3Mjg4MS4xNzQxOTEwNzMKMjAxNzgsMjQ5OTk2ODYuNjc2MzI3NzY1CjIwMTc5LDIyODg5NzgzLjAxMjY4MzM3CjIwMTgwLDM1MzM5MTk4LjA5MzQ3OTUKMjAxODEsNDQxMjcxNDcuMTMyMDUxODU1CjIwMTgyLDM0MDA1MjY0LjU0ODIwMTQ3CjIwMTgzLDI0NzUzNDAxLjMyMTI2ODY0CjIwMTg0LDMyNTg0MjQ4LjkxODQwMTM4NwoyMDE4NSwyODI4ODQyNC4zMzM3NTA1ODMKMjAxODYsMjMxNzg3ODAuOTg3NjkwMTY2CjIwMTg3LDE4MTkyMTUwLjY2MTMxNDE2CjIwMTg4LDE5MDcyNTY3Ljg0MTIxMjY2CjIwMTg5LDM0NTM1MDI1LjI1Nzc2NDM2CjIwMTkwLDQzMzA2Mzk0LjczMDE0MjA2CjIwMTkxLDE4MTc2MzEyLjI3MTcyNzc0CjIwMTkyLDMzODgzMTcyLjY2NjkzNjEzCjIwMTkzLDI3MjQ0NjQ3LjA1ODg1MTQ3CjIwMTk0LDM2MjUxMjY5LjU0MDc3NzQyCjIwMTk1LDI2MTI3MjQ3LjQ1NjYzMTEyNAoyMDE5NiwyMzE0Nzc3Mi42ODY4MTAzNzQKMjAxOTcsMzE4NTQ1MzIuOTEyMTE5NjQ2CjIwMTk4LDIwOTU5MTIxLjI0MTI5MzExNwoyMDE5OSwzMzExNzk1MS4xODE0OTY5NTUKMjAyMDAsNDAyMDU1MjMuNTczMjQ5MDMKMjAyMDEsMzMzMTk5MDQuNDA0Nzk1NTY1CjIwMjAyLDMzNjU0MTQ0LjY4OTYwMjk4NgoyMDIwMywyMzEyMDc5MC4xOTQyNDY2MTYKMjAyMDQsMjY3NDEzMDkuODYxOTAyOTc1CjIwMjA1LDIwMDQxOTQ4LjQxOTgyODkyCjIwMjA2LDM5Njk4NDA3LjUxMjgyNDcKMjAyMDcsMzI1MjIxOTAuNTQ5MzM2NDIKMjAyMDgsMzE1MTQxOTEuNTkyNTkxMDkyCjIwMjA5LDMxOTI4OTM3LjMwNDg4ODQxNgoyMDIxMCwyMjc0NDMyOC44NjM4MDc0CjIwMjExLDIyMDg0OTM4LjE2NzQ3NDkwMwoyMDIxMiwzMTI0OTYxNC42MjI3MDA2NzMKMjAyMTMsMzc4Mjc3NjcuOTIxMjM5NTUKMjAyMTQsMjAwOTEwNzguMDcyNDE0OAoyMDIxNSwyMDgzODM0Ni43NDQ2MDMxNQoyMDIxNiwxNzEzMDgyNC4yMDAxNjM4MzQKMjAyMTcsMzU5MjkxNTkuNzE1NzEwNDIKMjAyMTgsNDI3MzgzNjYuNjU5MzgzODcKMjAyMTksNDI4ODE3ODcuNTAwOTU0NzUKMjAyMjAsMzYyNDc5MzYuNTEyNjQ2NjQKMjAyMjEsMjU0MzA1MjcuOTA4MDM1NjczCjIwMjIyLDEzMjYyMzU2LjYwNjc4ODQwOAoyMDIyMywyMjc2NjcwNC41MzM0MzU2OTUKMjAyMjQsMjQ2NTc5MDUuNDMzMjE1NjEKMjAyMjUsNDAxMjYwNTAuNjA0NzEwOTc0CjIwMjI2LDI5NjU0MjIxLjQ0OTM5NTg5CjIwMjI3LDM2MTk4MzU3Ljk5NzE1MjgKMjAyMjgsMjg5MTM0MzYuODk4ODA0Mzk2CjIwMjI5LDMzNTcwNjU4LjQ3NjcwNTA1CjIwMjMwLDMyMzY3MzMyLjkxNzgyMzU5CjIwMjMxLDI2Nzg4ODMwLjUzODM0ODU2CjIwMjMyLDM2ODQ3NTQ5LjkzNTc4ODQxCjIwMjMzLDM5MDMzNzY3LjgyODA3MDUKMjAyMzQsMzc4MTM5NzIuMTgxNTQ3ODEKMjAyMzUsMzI1Njg2NDUuMTAwODk0ODk0CjIwMjM2LDI5NDkxNzM1Ljk2OTY2MDIwNAoyMDIzNyw0MDQ2OTExMy4yNjcxMTA3MwoyMDIzOCwyOTQ2MTEyMS4yMzg3NjMzMzIKMjAyMzksMjY3NDMyMjIuNDcyMTcwNTYKMjAyNDAsNDY5MzQ2NzQuNDE4NTg0NDIKMjAyNDEsMTc5ODI1NTUuMzExNjY1OTIKMjAyNDIsMjU5OTAyMzUuMDkxNTgyMjEzCjIwMjQzLDQyMDc1NDUyLjY5OTQyMzEzCjIwMjQ0LDE3NDMwMzQxLjY4NjMzMjg2MwoyMDI0NSwzMjUwODcyOS4xODAxNjA4MQoyMDI0NiwyMjg2OTU5OS40NzI5ODk1NTUKMjAyNDcsMTYwMzU1MjYuOTkwMzkwMjUKMjAyNDgsMzI5NjI5NjEuNTAwMzUwNzQ0CjIwMjQ5LDIyODM1MTIyLjgxODc0MjQ5NQoyMDI1MCwzNTMzNjg1Ni43MjYwNzM5NgoyMDI1MSwzNzM3NDk4My4zMjA0NjI3CjIwMjUyLDQ0MTAzNDAyLjQwMTc1MTc2CjIwMjUzLDIzNzM5MTk4LjY3MTczMTkwNAoyMDI1NCwzMzIxODM4NC42NzY0OTQ0OAoyMDI1NSwyNTM2ODM5My4wNzIzNDA1NjMKMjAyNTYsMjgwNjA5MjAuNDk2ODY4NTUKMjAyNTcsMTk5ODQxMjIuODIzNzg5NDQ4CjIwMjU4LDM0Nzc1MDA5LjI3NDQxNzQ0CjIwMjU5LDQ3NTU2MzA5LjQxNjg0OTkyNgoyMDI2MCwyMTY5OTUyMC43ODQzNjk4MjMKMjAyNjEsMzUxODE1NTcuNjMxNzQ1MzEKMjAyNjIsMTY4NjYzNTkuMDYyOTY4NzcyCjIwMjYzLDQyMDA1MTQwLjMyMjIyOTYyCjIwMjY0LDIwOTQyNjIyLjA2NDM0NjM3NwoyMDI2NSwyNzc0NDcxMC4xMzE0NTYwNzMKMjAyNjYsMjQ5OTkxNjAuMDM4ODQxNzUKMjAyNjcsMjY5OTg1NDEuNzgwMTU0MjkKMjAyNjgsMjMzMDcxNjUuMTU1Njc0MTkKMjAyNjksMzE3MjQ3MzIuMTAwNjQyNTE3CjIwMjcwLDMyNTYzNTgyLjgyODcxMTI0CjIwMjcxLDM5NjY1MDAzLjMyNDg0NTU4CjIwMjcyLDMwNjE0ODM2Ljk0NDE0OTc4CjIwMjczLDI0MDMzNDkwLjE0NjE4MjYyNgoyMDI3NCwyODQwNTIzNC42NTg4ODUwODQKMjAyNzUsMzQyMjc0NDYuNTAyOTQzNTkKMjAyNzYsMzg2NTg2OTQuNjc2NjU1NzI1CjIwMjc3LDE4ODQ2MDU0LjkwNzc0MzUzNgoyMDI3OCwzMzgwMzg0My43MjMyODU1OQoyMDI3OSwzMjczMzE5Ny41NDY3MzU3ODYKMjAyODAsNDEyMzU5ODMuMjE1NTM4NDM1CjIwMjgxLDIyNzI2NTIwLjE0MjA3ODE0NgoyMDI4MiwzMzU1OTI2MC42MTA5MDM5NgoyMDI4MywyNDA3MjM5NC41MzAwOTc1MDMKMjAyODQsMjcxNzM3ODMuNTMwNjI3MzA3CjIwMjg1LDE0MjYzNzkwLjgyNjA1OTQ4MwoyMDI4NiwzMzk5NjgwOS42NTkyNDUzMzUKMjAyODcsMTk2NTA1MDAuMTI1MTUyODg2CjIwMjg4LDI2MDg0NDA4Ljg2NTA2ODU0CjIwMjg5LDI2Njc2MDQxLjc0MzIyNzA5OAoyMDI5MCw0MzEyMTM3My45Mjk1MDA1NjUKMjAyOTEsMzk4NDMwMTkuNDY5NTAzMTkKMjAyOTIsMzU4MzI4NTYuNzc1Mjk2MDQKMjAyOTMsMjc5ODk2MjAuNTc4OTY1NjgzCjIwMjk0LDM0NDQ2MDU0Ljk4Mzk3MzQKMjAyOTUsMzExNDQ4MTEuODE3NDU3ODQ3CjIwMjk2LDI0NjM0NTQ4Ljk5ODI3MzkxMwoyMDI5NywxNjgyMzk3NC43NDkyNzYwMQoyMDI5OCwyNDc2Mzg0Ni43ODk3OTc0NDgKMjAyOTksMzk4NTYxNDguMDk2NzUwNTYKMjAzMDAsMjkzNzkxMDUuNzQ4MDQ1NDU2CjIwMzAxLDE4Njk5MTU2LjU3Nzc5NzQ3CjIwMzAyLDM0MDExNzYwLjg0NjAxMjA1CjIwMzAzLDI0MjM5MzEyLjc0NDYxOTE5NAoyMDMwNCwyMzI0ODE1My4wNTgzMDM4MTQKMjAzMDUsMjk5MDE5MDcuMjE1NTgzNzgzCjIwMzA2LDIxNzE0Njg4LjQ4NzUyMDk0OAoyMDMwNywzNTk1NjE2MS41OTE3NTY0ODUKMjAzMDgsMjIwNTQzMTMuNDk0Njg5NjMyCjIwMzA5LDIzODUwODcwLjY0NjI0ODY1NwoyMDMxMCwyMDQ1NjEwNi4wNDM3NDQ0MzcKMjAzMTEsMjA3NDU5MDMuMzcyMzQwOTkKMjAzMTIsMzczMzMyNDYuNDkyMjk0MgoyMDMxMyw0MTUzNDMxOC45NDg1MDk2OQoyMDMxNCwyMzM2ODA4NS4xOTQ3MjI5ODQKMjAzMTUsMjM0OTI1MDcuNTk3Nzg3Mjk1CjIwMzE2LDIwODQzMDgyLjQ0NDk1Njg0NwoyMDMxNywyMzU0MTE1MS43MTQ5MDE1MzcKMjAzMTgsMjcxMDY4NDkuOTA1NDQwMDQ3CjIwMzE5LDI3ODM3NDU0LjAwNzYxNzM4NAoyMDMyMCwxODA2ODE1MS41NzQ0ODAxOTUKMjAzMjEsMzc3NDA5ODguNDE4OTkxMTUKMjAzMjIsMzQzODg5NjkuNDU5NTIyOTgKMjAzMjMsMjkwOTE1NDMuOTE0MzU4MjQKMjAzMjQsNDEzNTgwNDUuMDE3MjI3NDkKMjAzMjUsMjQyOTM4NTcuNDc3NDUwNTU3CjIwMzI2LDM4MTU0MjU0LjMzMjQ2NTYwNAoyMDMyNywyNTY3OTUwNC40ODIyNjUwNjYKMjAzMjgsMTc4MTE2MzUuNTEyNzIxNzQ3CjIwMzI5LDIwMzMyMDAxLjYwODA4MjYwNAoyMDMzMCwzMjU0MzY4NC40MzA0MzkzNjgKMjAzMzEsMjYyMDg4ODMuNDg2NTM4MjkKMjAzMzIsMzMxODQ2ODMuMjM0MDc3MTcKMjAzMzMsNDIzNjMwNTQuMzA4ODU2NjE0CjIwMzM0LDQ3MTMyOTkwLjkyODc0MDQxCjIwMzM1LDI1NjA5MjUwLjM4NDI5ODE2CjIwMzM2LDI4NTczNTQ3LjEzMDc2NjQ0NAoyMDMzNywzNjQ0Mjc3Mi4wODAwNTcwNAoyMDMzOCwyMTI2MjQ2OS4yNTIxMTI5ODUKMjAzMzksNDY4NTg4MjcuMzQzMTE1MzQ1CjIwMzQwLDM2Mjk2Njc4Ljg5ODI3NTY0CjIwMzQxLDM5MDAxNjIyLjEyNTA2MjY5CjIwMzQyLDE4MTQ3NTk1Ljg2OTA5Nzk4NQoyMDM0MywzNTg5NzUzNC42NjkwODgwOTUKMjAzNDQsMjY1MjU4NTEuMTI1ODA0MjgzCjIwMzQ1LDE3ODUzMTI1LjM0NDIwOTc5CjIwMzQ2LDQwMzY5MzcwLjEwNjY0MzUwNQoyMDM0NywzODgwNzkzNy4wNDQ0NTIyNwoyMDM0OCwzNTQxMjAzNS44MzU1MDIxNTUKMjAzNDksNDEyNTU0ODMuNzQzNDcxNjYKMjAzNTAsMjM1OTI4OTkuNDQ2OTU1NzYKMjAzNTEsMzAxMDgzMDIuMTcwNzM1OQoyMDM1MiwyNDQ3MTI0OS4yMTI4MzY0MwoyMDM1MywzNTUzNTgwNS41Mzk3ODI3NTUKMjAzNTQsMjUwODg2NzUuNTUyODM5NzgKMjAzNTUsMjI5NDA3MjMuOTg2OTIwMTkzCjIwMzU2LDIzNDkwMjcwLjc2MTg1MDg4NgoyMDM1Nyw0ODQ4Njg3MC40Mzk3OTExNQoyMDM1OCw0MDYyNDM5Ni4wMTgwNDkyOAoyMDM1OSwzMzYwMjYyMS4yOTUxMTQ1NAoyMDM2MCwyNzYwNTMxMi45Mzg4NzUyODgKMjAzNjEsMjY0OTE0NzkuNzY1OTkzNTg0CjIwMzYyLDI5ODAzNzcyLjYxNTc0NTc4MwoyMDM2MywyNjYwNzU5Ny44NjE1NjgyNTcKMjAzNjQsMzY4Nzg2OTYuNDc3MDExMjUKMjAzNjUsMTg0NzAwMzUuODk4ODc0OTc2CjIwMzY2LDI2NjY2MDk5LjczNDMyOTMxMwoyMDM2NywyMjgzNjE4Mi45NjE3NzU0MjYKMjAzNjgsMjY2ODgzMTIuMDI5MTk0OTMKMjAzNjksNDY4NzA0NDEuNTczMjQ2MDIKMjAzNzAsMzU1ODAzNTguMzc5NDIyODY2CjIwMzcxLDQ4NzIwMDA5LjM4MDczNDI4CjIwMzcyLDM5NDA3ODQwLjczMDI0MDc0CjIwMzczLDMxNzI2OTgwLjA4NTM2OTIxCjIwMzc0LDIyNDYzODUyLjA2OTgxMzQ0CjIwMzc1LDI5MjQ4MTAyLjUwNzE0OTg1CjIwMzc2LDIwNjg5OTIyLjEwMjUzOTE0NAoyMDM3NywzMTYzMDMyMS4yNzk3MDE2NTQKMjAzNzgsMzkyMDU4NjMuMDE2MTA0OTQ0CjIwMzc5LDI5MzM1OTg5LjAxMzAyNzUxNQoyMDM4MCwyNzMzMDMwNC40OTIyNDA3OQoyMDM4MSw0Njk3MTQyNC4xMTczNTgwMwoyMDM4MiwyMTczODg5Ny41NTc5NzQyNTMKMjAzODMsMzMxMTQxNzAuMTM4Mjk1MjYKMjAzODQsNTM1ODYwMzUuNDgyMzcxOAoyMDM4NSwxOTY0NDE2My4xOTI1MjUyNgoyMDM4NiwyODUyMDMzOS4wODkxOTk1MwoyMDM4NywyMzg2OTg4MS41NTIxMQoyMDM4OCwxODk2MTM1NC4xOTMxMjYyMgoyMDM4OSwzMDE4MTg2Ny4xODE0NDU5MzQKMjAzOTAsNTk5MTIwNDAuNDUwNDc5MzcKMjAzOTEsMjAyMDM1MzAuODkwMjQ3NzE0CjIwMzkyLDQ2MTMyNjU0Ljg2NTc5MzM2CjIwMzkzLDIzMzQ4NTE1LjM3OTA0NTA0MwoyMDM5NCwxNDE1Mjg3My40NjQwNjA1NDcKMjAzOTUsMzkyMjQ1NzAuMTYwNzg0MDM2CjIwMzk2LDI5NjYyMjU0LjY2MzA2NTcyNAoyMDM5NywzMTMxNTA1Ny4xMjgwMDA1NTQKMjAzOTgsMzM3MDQ3ODkuNDM2NzI4MDkKMjAzOTksMzAwNzQwNDIuODIyMTE1MzE3CjIwNDAwLDIyNTg4NzUzLjcxOTQwNTMzCjIwNDAxLDMwNTQyNTQ3Ljg0NzAyNjk5CjIwNDAyLDI5NzYwNzk5LjQ2NDM0MjE3MwoyMDQwMywzNTA1NjU3OC43OTA1MDUyNDUKMjA0MDQsMjEzMTYyNDUuNTg4MjkzNDYKMjA0MDUsMzc2OTE4MDcuODE4NDMwNjkKMjA0MDYsNDQ3MDgyMDcuNzYzODg0ODQKMjA0MDcsMTcwOTIwMzYuNzU3MTY3MTYKMjA0MDgsMjI4MjY0MDUuMTc3MzcxODIyCjIwNDA5LDMzMjM5ODYxLjY0MTE5NDMzCjIwNDEwLDE5MzA2ODYyLjcwNTQ3NDYzNAoyMDQxMSwyMjMyOTc1Ni4xNDgxODc3MzgKMjA0MTIsMzE5MDEyMTguNDE4MDEzOTgKMjA0MTMsMjMxNjA4NjIuMTEyMjk3MDk1CjIwNDE0LDI2MDkxNDg5LjA1MzU5MDY5MwoyMDQxNSwyMzQ3Nzk0OS4zMTAxODcxNgoyMDQxNiwzMDg1MjY4MS43NzkwNDAwNzYKMjA0MTcsMjYzODA4MTEuODQyNDI1NzcKMjA0MTgsMjcwMzI0MTguNjEwMjY5MDAzCjIwNDE5LDE5MjM4NTAwLjU2MjQ5OTQ4MgoyMDQyMCwzMjM3MzE5My4wMjMxODY3OAoyMDQyMSwzMDY5MjcwOS40MDQ3ODMxMDQKMjA0MjIsMzc0MzA1MjEuMjc0NzkzNTgKMjA0MjMsMzM5MTI3MTAuOTk0MzQxNTQKMjA0MjQsMzE3MzAzMjIuODEyMTI3MTIKMjA0MjUsMjAyMDg5MDUuNDA4MzgyODMzCjIwNDI2LDI5Njk4ODU5LjUyMDY4NTA2CjIwNDI3LDI1NDg3MDY0LjQwOTAzMjM0NQoyMDQyOCwzNzM5MTE4NS4wNjg2NTc3MgoyMDQyOSwzMTYyMTg2OS42NzYwNDU4NwoyMDQzMCwxNzcyMzkwNC40MjA2Nzk1MTcKMjA0MzEsMjU2MjgzOTAuMTYzNjkwMjY1CjIwNDMyLDIzMjE3NjE0LjUwNjE0MzQ1NAoyMDQzMywzODcyMjM5Ny4xNDYxNjE2NwoyMDQzNCwyMDMyODIyNy4xMDA2ODc3NAoyMDQzNSwzMjA0ODM0MC4zMTgwMDU4NjcKMjA0MzYsNDMxNzY3MzIuMTkxMzI2MTUKMjA0MzcsMzg1ODQ1MzAuMjExNTM3MTUKMjA0MzgsNDE4NzMwMDUuMDEwODE2NTE0CjIwNDM5LDM4MjkwOTEzLjYyMzAzMjE4CjIwNDQwLDIxMzkyNzQ1LjMxODIyNTg1MwoyMDQ0MSwyNDM4MDgxNC4wNzg1MzEzMQoyMDQ0MiwyOTcwMTQ0NC43MzYzMDA4NjMKMjA0NDMsMTg5MzM2MDEuNTYzMzM4MjcKMjA0NDQsMzA2ODcwNzguMTU4MDI5ODU0CjIwNDQ1LDI0Mzc1ODU5LjUzMjIxMDkxCjIwNDQ2LDI0NDA2NTU5LjEzMTU2OTEyCjIwNDQ3LDE5NTAzMTkwLjE3ODE3NjkKMjA0NDgsMjAzMjA5MDEuMDA5MTUzNTgKMjA0NDksNDQyOTY2NDIuOTIzNjI5CjIwNDUwLDI5Njg2NTc4Ljg2NzI2MjcxNAoyMDQ1MSwzNTcyNzI5OS4zMDI0MzQ0NDQKMjA0NTIsMTg5MDMxMDUuMjM5NTQ4NTI3CjIwNDUzLDIyMjExMjYzLjk2MTczNzQ2NQoyMDQ1NCwyOTIzNTkzNi4zODY3NDA5NjgKMjA0NTUsNDYzNDg3MDQuMTY5NjgzODQ0CjIwNDU2LDE0OTAwMjUwLjM4MjEyMjEyMgoyMDQ1Nyw0NjE1NjY0NS4xOTU2MTg5NAoyMDQ1OCwyODg0NTUxNy40NTM5NDQ4NDcKMjA0NTksMjg0NzMyMzQuNTc3NTM2ODE4CjIwNDYwLDI3ODk0MTYyLjI0MDIyOTY4NQoyMDQ2MSwzOTU3NzExNi4wMjQzNjc3OAoyMDQ2MiwxODI0OTQ5OS4xNzU5NzUzMjYKMjA0NjMsMzc1MTUzMDguNzE4NzgwMzYKMjA0NjQsMTk5NzAxODguMjkwNjcxNDk4CjIwNDY1LDQwMDA3ODQ2LjQzOTk4MjYxCjIwNDY2LDQwMjMzNzcxLjEwOTM2Mzk3CjIwNDY3LDE2MTQyNTIwLjA2OTU0MTcxMQoyMDQ2OCwxNzIxMjczOC43MzE3NTIxOQoyMDQ2OSw0MDAxMzIwMC4yMDg3NTY4NwoyMDQ3MCwyODc5NDI3OC40OTY2ODY3MQoyMDQ3MSwyNzkxMjU3OC41Mjg1OTQxOTIKMjA0NzIsMzQ4NDAyNjguNjI4Nzc1NTgKMjA0NzMsNDE0MjM0ODAuMzkwODY3NDQKMjA0NzQsMTk1MzE5NTQuNDUwNTU4ODIzCjIwNDc1LDMzNTgwODExLjYxNjQ3MTg0CjIwNDc2LDExNTU0MjgyLjU3MjAyMjgwNwoyMDQ3NywyOTY1ODkyNi42MzU3MjExNAoyMDQ3OCwzNjA3MzIyNC4wMDI0MzE2NQoyMDQ3OSw0MTkyNzU1Ni41Mzk5OTcxNwoyMDQ4MCwzMzM2MDE2NS42MTUzODU3MzQKMjA0ODEsMzcwMjQ2NzcuOTY1MzUxMzIKMjA0ODIsMzQ3MzM2MDcuNzY3OTc5MzIKMjA0ODMsMzQ3NTY5ODUuNzE0Mzk4ODE2CjIwNDg0LDQ4OTYzNTAyLjU1NDQzOTMxCjIwNDg1LDI4NzU5MzkxLjE4MTI2NDkwNwoyMDQ4NiwxNDQ3NDY0MC4zMjAwMDU4MjUKMjA0ODcsMTgxMzgzMjcuODMzOTg3MzQ4CjIwNDg4LDMwMDAxMjEyLjczMTg5MjA1CjIwNDg5LDI2NDUzODUxLjcyOTE3Njk1MwoyMDQ5MCwyNTAxNzYxMC41NDAzNzYzODQKMjA0OTEsMzA3NjAwNzUuMDk3MTI2NTQ0CjIwNDkyLDI5NDUyNjE3LjA5Njg1MDI4NwoyMDQ5MywyNzAyNzgyOC45OTQ1NTM5MjcKMjA0OTQsMjc5MTc5MDguMjYxOTY4NDI2CjIwNDk1LDMxMzc1NjE5LjkzMzYxMDkyNwoyMDQ5NiwzMjU2ODQ2NS42OTM2MDQ2OTMKMjA0OTcsMjk4NzQ3MjkuODMwODQ2NTE1CjIwNDk4LDM0MDg3Mjk3LjU4OTkyNjU3CjIwNDk5LDQyNzQyMzEwLjA3NzU1Mzc2CjIwNTAwLDI3NTcxNDk5LjIwMTE3OTM2Cg=="}')

WORK_ROOT = OUTPUT_DIR / 'improvement_v7'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for name, source in SOURCE_FILES.items():
    (WORK_ROOT / name).write_text(source)
backup = WORK_ROOT / 'scored_v5_backup'
backup.mkdir(exist_ok=True)
for name, payload in PRESERVED_V5.items():
    (backup / name).write_bytes(base64.b64decode(payload))
print('Research root:', WORK_ROOT)
print('Resume checkpoints:', WORK_ROOT / 'results/cache')

## Execute the bounded research

Each stage is a subprocess using the embedded source. Completed folds are reused
only with matching inputs, configurations, sources and file hashes. Interrupted
partial writes are recomputed. No additional model settings are added after the
shortlist is frozen. Depending on CPU availability, nested residual work can take
substantially longer than the simple models.

In [ ]:
subprocess.run([sys.executable, str(WORK_ROOT / 'prepare_resume.py')], check=True)
subprocess.run([sys.executable, str(WORK_ROOT / 'workflow.py'), 'audit', '--data-dir', str(DATA_DIR)], check=True)

In [ ]:
subprocess.run([sys.executable, str(WORK_ROOT / 'prepare_resume.py')], check=True)
subprocess.run([sys.executable, str(WORK_ROOT / 'workflow.py'), 'screen', '--data-dir', str(DATA_DIR)], check=True)

In [ ]:
subprocess.run([sys.executable, str(WORK_ROOT / 'prepare_resume.py')], check=True)
subprocess.run([sys.executable, str(WORK_ROOT / 'workflow.py'), 'confirm', '--data-dir', str(DATA_DIR)], check=True)

In [ ]:
subprocess.run([sys.executable, str(WORK_ROOT / 'prepare_resume.py')], check=True)
subprocess.run([sys.executable, str(WORK_ROOT / 'workflow.py'), 'fit', '--data-dir', str(DATA_DIR)], check=True)

In [ ]:
decision = json.loads((WORK_ROOT / 'results/decision.json').read_text())
print(json.dumps(decision, indent=2))
print('Actual portal scores remain unknown until the experimental CSVs are uploaded.')
import zipfile
archive = OUTPUT_DIR / 'V7_research_results.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for path in WORK_ROOT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts:
            z.write(path, path.relative_to(OUTPUT_DIR))
try:
    from google.colab import files
except ImportError:
    print('Saved research bundle:', archive)
else:
    files.download(str(archive))